In [ ]:
import torch as torch
import torch.nn as nn
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

In [ ]:

# Standard GroupNorm + Swish
class StandardNormActivation(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(num_groups=max(4, min(8, channels//4)),
                                num_channels=channels)
        self.activation = nn.SiLU()  # Swish

    def forward(self, x):
        return self.activation(self.norm(x))

# Enhanced version with channel attention
class GroupMinMaxDoubleSwish(nn.Module):
    def __init__(self, eps=1e-6, residual_weight=0.1):
        super().__init__()
        self.eps = eps
        self.residual_weight = residual_weight

    def forward(self, x):
        B, C, H, W = x.shape
        original_x = x

        num_groups = max(4, min(8, C // 4))
        x_grouped = x.view(B, num_groups, C // num_groups, H, W)

        group_min = torch.amin(x_grouped, dim=(2, 3, 4), keepdim=True)
        group_max = torch.amax(x_grouped, dim=(2, 3, 4), keepdim=True)
        range_ = (group_max - group_min).clamp(min=self.eps)
        x_normalized = x_grouped / range_
        x_normalized = x_normalized.view(B, C, H, W)

        # Double Swish: 2x/(1+e^{-x})
        activated = 2 * x_normalized * torch.sigmoid(x_normalized)

        return (1 - self.residual_weight) * activated + self.residual_weight * original_x
class EnhancedDoubleSwish(nn.Module):
    def __init__(self, channels, eps=1e-6, residual_weight=0.1):
        super().__init__()
        self.base_activation = GroupMinMaxDoubleSwish(eps, residual_weight)
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(4, channels//4), 1),
            nn.ReLU(),
            nn.Conv2d(max(4, channels//4), channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        base = self.base_activation(x)
        weights = self.channel_gate(x)
        return base * weights


In [ ]:

# Data generator function
def generate_medical_like_data(batch_size=1, channels=4, size=64, sparsity=0.85, high_std=5.0):
    data = torch.zeros((batch_size, channels, size, size))
    for b in range(batch_size):
        for c in range(channels):
            non_zero_mask = torch.rand(size, size) > sparsity
            num_non_zero = non_zero_mask.sum()

            if num_non_zero > 0:
                values = torch.abs(torch.randn(num_non_zero) * high_std)
                if c % 2 == 0:  # Create structured patterns
                    values = values * (1 + 0.5 * torch.sin(torch.linspace(0, 3*np.pi, num_non_zero)))
                data[b, c] = data[b, c].masked_scatter(non_zero_mask, values)
    return data


In [ ]:
def comprehensive_comparison():
    torch.manual_seed(42)
    data = generate_medical_like_data()

    # Calculate metrics for original data
    non_zero_mask = (data > 0)
    data_non_zero = data[non_zero_mask]
    original_metrics = {
        "output_mean": data.mean().item(),
        "output_std": data.std().item(),
        "sparsity": (data == 0).float().mean().item(),
        "non_zero_preservation": 1.0,  # Reference value
        "dynamic_range": data.max().item() - data.min().item(),
        "min_val": data.min().item(),
        "max_val": data.max().item(),
    }

    # Initialize all methods
    methods = {
        "Standard (GroupNorm+Swish)": StandardNormActivation(channels=4),
        "GroupMinMax+DoubleSwish": GroupMinMaxDoubleSwish(),
        "Enhanced (ChannelAttn)": EnhancedDoubleSwish(channels=4),
    }

    # Process data through all methods
    results = {}
    outputs = {}
    for name, model in methods.items():
        out = model(data)
        outputs[name] = out.detach()

        # Calculate metrics
        out_non_zero = out[non_zero_mask]

        results[name] = {
            "output_mean": out.mean().item(),
            "output_std": out.std().item(),
            "sparsity": (out.abs() < 1e-3).float().mean().item(),
            "non_zero_preservation": out_non_zero.mean().item() / data_non_zero.mean().item(),
            "dynamic_range": out.max().item() - out.min().item(),
            "min_val": out.min().item(),
            "max_val": out.max().item(),
        }

    # Print metrics in copyable format
    print("\n" + "="*80)
    print("Quantitative Metrics Comparison (Copyable Format)")
    print("="*80)

    # Header
    print(f"{'Method':<25} | {'Mean':>8} | {'Std':>8} | {'Sparsity':>8} | {'NZ Pres':>8} | {'Min':>8} | {'Max':>8} | {'Dyn Range':>9}")
    print("-"*95)

    # Original data
    print(f"{'Original Data':<25} | "
          f"{original_metrics['output_mean']:8.4f} | "
          f"{original_metrics['output_std']:8.4f} | "
          f"{original_metrics['sparsity']:8.4f} | "
          f"{'1.0000':>8} | "  # Non-zero preservation is 1.0 by definition
          f"{original_metrics['min_val']:8.4f} | "
          f"{original_metrics['max_val']:8.4f} | "
          f"{original_metrics['dynamic_range']:9.4f}")

    # Method results
    for name in methods:
        res = results[name]
        print(f"{name:<25} | "
              f"{res['output_mean']:8.4f} | "
              f"{res['output_std']:8.4f} | "
              f"{res['sparsity']:8.4f} | "
              f"{res['non_zero_preservation']:8.4f} | "
              f"{res['min_val']:8.4f} | "
              f"{res['max_val']:8.4f} | "
              f"{res['dynamic_range']:9.4f}")

    print("="*80 + "\n")

    # ... [rest of visualization code remains the same] ...

    return results

# Run the comparison
results = comprehensive_comparison()


Quantitative Metrics Comparison (Copyable Format)
Method                    |     Mean |      Std | Sparsity |  NZ Pres |      Min |      Max | Dyn Range
-----------------------------------------------------------------------------------------------
Original Data             |   0.6302 |   1.9955 |   0.8492 |   1.0000 |   0.0000 |  21.8818 |   21.8818
Standard (GroupNorm+Swish) |   0.1340 |   0.9175 |   0.0001 |   0.3926 |  -0.1415 |  10.1890 |   10.3305
GroupMinMax+DoubleSwish   |   0.1020 |   0.3278 |   0.8493 |   0.1618 |   0.0000 |   3.5041 |    3.5041
Enhanced (ChannelAttn)    |   0.0531 |   0.1733 |   0.8494 |   0.0843 |   0.0000 |   1.9956 |    1.9956



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.optim import Adam
import time

# Your activation functions
class GroupMinMaxDoubleSwish(nn.Module):
    def __init__(self, eps=1e-6, residual_weight=0.1):
        super().__init__()
        self.eps = eps
        self.residual_weight = residual_weight

    def forward(self, x):
        B, C, H, W = x.shape
        original_x = x
        num_groups = max(4, min(8, C // 4))
        x_grouped = x.view(B, num_groups, C // num_groups, H, W)
        group_min = torch.amin(x_grouped, dim=(2, 3, 4), keepdim=True)
        group_max = torch.amax(x_grouped, dim=(2, 3, 4), keepdim=True)
        range_ = (group_max - group_min).clamp(min=self.eps)
        x_normalized = x_grouped / range_
        x_normalized = x_normalized.view(B, C, H, W)

        activated = 2 * x_normalized * torch.sigmoid(x_normalized)
        return (1 - self.residual_weight) * activated + self.residual_weight * original_x

class EnhancedDoubleSwish(nn.Module):
    def __init__(self, channels, eps=1e-6, residual_weight=0.1):
        super().__init__()
        self.base_activation = GroupMinMaxDoubleSwish(eps, residual_weight)
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(4, channels//4), 1),
            nn.ReLU(),
            nn.Conv2d(max(4, channels//4), channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        base = self.base_activation(x)
        weights = self.channel_gate(x)
        return base * weights

# Simple test models
class SimpleSegModel(nn.Module):
    def __init__(self, activation_type='standard', channels=16):
        super().__init__()
        self.conv1 = nn.Conv2d(4, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv3 = nn.Conv2d(channels, 4, 3, padding=1)

        if activation_type == 'standard':
            self.norm1 = nn.GroupNorm(4, channels)
            self.norm2 = nn.GroupNorm(4, channels)
            self.act = nn.SiLU()
        elif activation_type == 'groupminmax':
            self.norm1 = nn.Identity()
            self.norm2 = nn.Identity()
            self.act = GroupMinMaxDoubleSwish()
        else:  # enhanced
            self.norm1 = nn.Identity()
            self.norm2 = nn.Identity()
            self.act = EnhancedDoubleSwish(channels)

    def forward(self, x):
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.act(x)

        x = self.conv2(x)
        x = self.norm2(x)
        x = self.act(x)

        x = self.conv3(x)
        return x

def create_sparse_data(batch_size=8, size=64, sparsity=0.85):
    """Create synthetic data matching your scenario"""
    # 4 input channels with 85% sparsity
    data = torch.randn(batch_size, 4, size, size)
    mask = torch.rand_like(data) < sparsity
    data[mask] = 0

    # Create labels with 97-99% background (class 0)
    labels = torch.zeros(batch_size, size, size, dtype=torch.long)

    for b in range(batch_size):
        # Add small patches of other classes
        for class_id in [1, 2, 3]:
            if torch.rand(1) < 0.7:  # Sometimes have this class
                # Small random patches
                num_patches = torch.randint(1, 4, (1,)).item()
                for _ in range(num_patches):
                    h_start = torch.randint(0, size-8, (1,)).item()
                    w_start = torch.randint(0, size-8, (1,)).item()
                    h_size = torch.randint(2, 6, (1,)).item()
                    w_size = torch.randint(2, 6, (1,)).item()
                    labels[b, h_start:h_start+h_size, w_start:w_start+w_size] = class_id

    return data, labels

def test_activation_methods():
    """Test different activation methods on sparse segmentation task"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Create test data
    train_data, train_labels = create_sparse_data(32, 64)
    test_data, test_labels = create_sparse_data(16, 64)

    print(f"Input sparsity: {(train_data == 0).float().mean():.3f}")
    print(f"Background ratio: {(train_labels == 0).float().mean():.3f}")

    methods = ['standard', 'groupminmax', 'enhanced']
    results = {}

    for method in methods:
        print(f"\n=== Testing {method.upper()} ===")

        model = SimpleSegModel(method).to(device)
        optimizer = Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        # Training loop
        model.train()
        losses = []
        start_time = time.time()

        for epoch in range(20):  # Quick training
            optimizer.zero_grad()

            # Forward pass
            outputs = model(train_data.to(device))
            loss = criterion(outputs, train_labels.to(device))

            # Backward pass
            loss.backward()

            # Track gradient norms for minority classes
            total_grad_norm = 0
            for param in model.parameters():
                if param.grad is not None:
                    total_grad_norm += param.grad.norm().item()

            optimizer.step()
            losses.append(loss.item())

            if epoch % 5 == 0:
                print(f"Epoch {epoch}: Loss {loss.item():.4f}, Grad norm: {total_grad_norm:.4f}")

        train_time = time.time() - start_time

        # Evaluation
        model.eval()
        with torch.no_grad():
            test_outputs = model(test_data.to(device))
            test_loss = criterion(test_outputs, test_labels.to(device))

            # Calculate per-class IoU
            preds = test_outputs.argmax(1)
            ious = []

            for class_id in range(4):
                pred_mask = (preds == class_id)
                true_mask = (test_labels.to(device) == class_id)

                intersection = (pred_mask & true_mask).float().sum()
                union = (pred_mask | true_mask).float().sum()

                if union > 0:
                    iou = intersection / union
                else:
                    iou = torch.tensor(1.0)  # Perfect if no true or pred pixels

                ious.append(iou.item())

            # Analyze activations in the model
            x = train_data[:4].to(device)  # Small batch for analysis
            x = model.conv1(x)
            x = model.norm1(x)
            activations = model.act(x)

            activation_stats = {
                'mean': activations.mean().item(),
                'std': activations.std().item(),
                'sparsity': (activations == 0).float().mean().item(),
                'nonzero_ratio': (activations != 0).float().mean().item(),
                'max': activations.max().item(),
                'min': activations.min().item()
            }

        results[method] = {
            'final_loss': losses[-1],
            'test_loss': test_loss.item(),
            'class_ious': ious,
            'mean_minority_iou': np.mean(ious[1:]),  # Classes 1,2,3
            'background_iou': ious[0],
            'train_time': train_time,
            'activation_stats': activation_stats,
            'loss_curve': losses
        }

        print(f"Final train loss: {losses[-1]:.4f}")
        print(f"Test loss: {test_loss.item():.4f}")
        print(f"Class IoUs: {[f'{iou:.3f}' for iou in ious]}")
        print(f"Mean minority IoU: {np.mean(ious[1:]):.3f}")
        print(f"Training time: {train_time:.2f}s")

    return results

def analyze_results(results):
    """Analyze and compare results"""
    print("\n" + "="*80)
    print("COMPREHENSIVE COMPARISON")
    print("="*80)

    methods = list(results.keys())

    # Performance comparison
    print(f"{'Method':<15} {'Train Loss':<12} {'Test Loss':<12} {'Bg IoU':<10} {'Min IoU':<10} {'Time':<8}")
    print("-" * 75)

    for method in methods:
        r = results[method]
        print(f"{method:<15} {r['final_loss']:<12.4f} {r['test_loss']:<12.4f} "
              f"{r['background_iou']:<10.3f} {r['mean_minority_iou']:<10.3f} {r['train_time']:<8.1f}s")

    # Activation analysis
    print(f"\n{'Method':<15} {'Act Mean':<10} {'Act Std':<10} {'Sparsity':<10} {'NonZero':<10} {'Range':<10}")
    print("-" * 75)

    for method in methods:
        stats = results[method]['activation_stats']
        range_val = stats['max'] - stats['min']
        print(f"{method:<15} {stats['mean']:<10.4f} {stats['std']:<10.4f} "
              f"{stats['sparsity']:<10.3f} {stats['nonzero_ratio']:<10.3f} {range_val:<10.3f}")

    # Key insights
    print(f"\nKEY INSIGHTS:")
    best_minority = max(methods, key=lambda m: results[m]['mean_minority_iou'])
    fastest = min(methods, key=lambda m: results[m]['train_time'])
    most_stable = min(methods, key=lambda m: results[m]['test_loss'])

    print(f"Best for minority classes: {best_minority.upper()} (IoU: {results[best_minority]['mean_minority_iou']:.3f})")
    print(f"Fastest training: {fastest.upper()} ({results[fastest]['train_time']:.1f}s)")
    print(f"Most stable: {most_stable.upper()} (test loss: {results[most_stable]['test_loss']:.4f})")

    # Check if your method is actually useful
    your_method_minority = results['enhanced']['mean_minority_iou']
    baseline_minority = results['standard']['mean_minority_iou']

    print(f"\nYOUR METHOD VERDICT:")
    if your_method_minority > baseline_minority * 1.1:  # 10% improvement
        print(f"✅ USEFUL! {(your_method_minority/baseline_minority-1)*100:.1f}% improvement on minority classes")
    elif your_method_minority > baseline_minority * 0.9:  # Within 10%
        print(f"🤔 MARGINAL. Only {(your_method_minority/baseline_minority-1)*100:.1f}% change on minority classes")
    else:
        print(f"❌ NOT HELPFUL. {(1-your_method_minority/baseline_minority)*100:.1f}% worse on minority classes")

if __name__ == "__main__":
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)

    print("Testing activation methods on sparse segmentation task...")
    print("Input: 4 channels, 85% sparsity")
    print("Labels: 4 classes, 97-99% background")

    results = test_activation_methods()
    analyze_results(results)

Testing activation methods on sparse segmentation task...
Input: 4 channels, 85% sparsity
Labels: 4 classes, 97-99% background
Using device: cpu
Input sparsity: 0.851
Background ratio: 0.985

=== Testing STANDARD ===
Epoch 0: Loss 1.3564, Grad norm: 9.1576
Epoch 5: Loss 0.8470, Grad norm: 6.7002
Epoch 10: Loss 0.5249, Grad norm: 4.9323
Epoch 15: Loss 0.3233, Grad norm: 3.2077
Final train loss: 0.2292
Test loss: 0.2034
Class IoUs: ['0.988', '0.000', '0.000', '0.000']
Mean minority IoU: 0.000
Training time: 4.12s

=== Testing GROUPMINMAX ===
Epoch 0: Loss 1.3443, Grad norm: 2.7952
Epoch 5: Loss 1.2230, Grad norm: 2.7335
Epoch 10: Loss 1.0829, Grad norm: 3.2046
Epoch 15: Loss 0.8965, Grad norm: 3.7008
Final train loss: 0.7190
Test loss: 0.6819
Class IoUs: ['0.988', '0.000', '0.000', '0.000']
Mean minority IoU: 0.000
Training time: 5.43s

=== Testing ENHANCED ===
Epoch 0: Loss 1.3383, Grad norm: 2.1249
Epoch 5: Loss 1.2457, Grad norm: 2.3104
Epoch 10: Loss 1.1322, Grad norm: 2.6521
Epoch 1

/tmp/ipython-input-1124725617.py:261: RuntimeWarning: invalid value encountered in scalar divide
  print(f"❌ NOT HELPFUL. {(1-your_method_minority/baseline_minority)*100:.1f}% worse on minority classes")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.optim import Adam
import time

# Your activation functions
class GroupMinMaxDoubleSwish(nn.Module):
    def __init__(self, eps=1e-6, residual_weight=0.1):
        super().__init__()
        self.eps = eps
        self.residual_weight = residual_weight

    def forward(self, x):
        B, C, H, W = x.shape
        original_x = x
        num_groups = max(4, min(8, C // 4))
        x_grouped = x.view(B, num_groups, C // num_groups, H, W)
        group_min = torch.amin(x_grouped, dim=(2, 3, 4), keepdim=True)
        group_max = torch.amax(x_grouped, dim=(2, 3, 4), keepdim=True)
        range_ = (group_max - group_min).clamp(min=self.eps)
        x_normalized = x_grouped / range_
        x_normalized = x_normalized.view(B, C, H, W)

        activated = 2 * x_normalized * torch.sigmoid(x_normalized)
        return (1 - self.residual_weight) * activated + self.residual_weight * original_x

class EnhancedDoubleSwish(nn.Module):
    def __init__(self, channels, eps=1e-6, residual_weight=0.1):
        super().__init__()
        self.base_activation = GroupMinMaxDoubleSwish(eps, residual_weight)
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(4, channels//4), 1),
            nn.ReLU(),
            nn.Conv2d(max(4, channels//4), channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        base = self.base_activation(x)
        weights = self.channel_gate(x)
        return base * weights

# Fixed version for extreme imbalance
class FixedSparseActivation(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        # Much gentler normalization + stronger residual
        self.eps = eps
        self.residual_weight = 0.7  # Keep more of original signal

        # Simpler channel attention
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Gentler normalization per channel (not group)
        original_x = x
        x_norm = F.layer_norm(x, x.shape[1:])  # Preserve more variation

        # Lighter activation
        activated = x_norm * torch.sigmoid(x_norm)  # Single swish

        # Channel attention
        weights = self.channel_gate(x)
        gated = activated * weights

        # Strong residual connection
        return self.residual_weight * original_x + (1 - self.residual_weight) * gated
# Simple test models
class SimpleSegModel(nn.Module):
    def __init__(self, activation_type='standard', channels=16):
        super().__init__()
        self.conv1 = nn.Conv2d(4, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv3 = nn.Conv2d(channels, 4, 3, padding=1)

        if activation_type == 'standard':
            self.norm1 = nn.GroupNorm(4, channels)
            self.norm2 = nn.GroupNorm(4, channels)
            self.act = nn.SiLU()
        elif activation_type == 'groupminmax':
            self.norm1 = nn.Identity()
            self.norm2 = nn.Identity()
            self.act = GroupMinMaxDoubleSwish()
        elif activation_type == 'enhanced':
            self.norm1 = nn.Identity()
            self.norm2 = nn.Identity()
            self.act = EnhancedDoubleSwish(channels)
        else:  # fixed
            self.norm1 = nn.Identity()
            self.norm2 = nn.Identity()
            self.act = FixedSparseActivation(channels)

    def forward(self, x):
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.act(x)

        x = self.conv2(x)
        x = self.norm2(x)
        x = self.act(x)

        x = self.conv3(x)
        return x

def create_sparse_data(batch_size=8, size=64, sparsity=0.85):
    """Create synthetic data matching your scenario"""
    # 4 input channels with 85% sparsity
    data = torch.randn(batch_size, 4, size, size)
    mask = torch.rand_like(data) < sparsity
    data[mask] = 0

    # Create labels with 97-99% background (class 0)
    labels = torch.zeros(batch_size, size, size, dtype=torch.long)

    for b in range(batch_size):
        # Add small patches of other classes
        for class_id in [1, 2, 3]:
            if torch.rand(1) < 0.7:  # Sometimes have this class
                # Small random patches
                num_patches = torch.randint(1, 4, (1,)).item()
                for _ in range(num_patches):
                    h_start = torch.randint(0, size-8, (1,)).item()
                    w_start = torch.randint(0, size-8, (1,)).item()
                    h_size = torch.randint(2, 6, (1,)).item()
                    w_size = torch.randint(2, 6, (1,)).item()
                    labels[b, h_start:h_start+h_size, w_start:w_start+w_size] = class_id

    return data, labels

def test_activation_methods():
    """Test different activation methods on sparse segmentation task"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Create test data with MORE minority class presence
    train_data, train_labels = create_sparse_data(32, 64)
    test_data, test_labels = create_sparse_data(16, 64)

    # Force some minority classes to exist
    for b in range(len(train_labels)):
        # Add larger patches to make detection easier
        for class_id in [1, 2, 3]:
            h_start = torch.randint(5, 59, (1,)).item()
            w_start = torch.randint(5, 59, (1,)).item()
            train_labels[b, h_start:h_start+5, w_start:w_start+5] = class_id

    print(f"Input sparsity: {(train_data == 0).float().mean():.3f}")
    print(f"Background ratio: {(train_labels == 0).float().mean():.3f}")

    methods = ['standard', 'groupminmax', 'enhanced', 'fixed']
    results = {}

    for method in methods:
        print(f"\n=== Testing {method.upper()} ===")

        model = SimpleSegModel(method).to(device)
        optimizer = Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        # Training loop
        model.train()
        losses = []
        start_time = time.time()

        for epoch in range(20):  # Quick training
            optimizer.zero_grad()

            # Forward pass
            outputs = model(train_data.to(device))
            loss = criterion(outputs, train_labels.to(device))

            # Backward pass
            loss.backward()

            # Track gradient norms for minority classes
            total_grad_norm = 0
            for param in model.parameters():
                if param.grad is not None:
                    total_grad_norm += param.grad.norm().item()

            optimizer.step()
            losses.append(loss.item())

            if epoch % 5 == 0:
                print(f"Epoch {epoch}: Loss {loss.item():.4f}, Grad norm: {total_grad_norm:.4f}")

        train_time = time.time() - start_time

        # Evaluation
        model.eval()
        with torch.no_grad():
            test_outputs = model(test_data.to(device))
            test_loss = criterion(test_outputs, test_labels.to(device))

            # Calculate per-class IoU
            preds = test_outputs.argmax(1)
            ious = []

            for class_id in range(4):
                pred_mask = (preds == class_id)
                true_mask = (test_labels.to(device) == class_id)

                intersection = (pred_mask & true_mask).float().sum()
                union = (pred_mask | true_mask).float().sum()

                if union > 0:
                    iou = intersection / union
                else:
                    iou = torch.tensor(1.0)  # Perfect if no true or pred pixels

                ious.append(iou.item())

            # Analyze activations in the model
            x = train_data[:4].to(device)  # Small batch for analysis
            x = model.conv1(x)
            x = model.norm1(x)
            activations = model.act(x)

            activation_stats = {
                'mean': activations.mean().item(),
                'std': activations.std().item(),
                'sparsity': (activations == 0).float().mean().item(),
                'nonzero_ratio': (activations != 0).float().mean().item(),
                'max': activations.max().item(),
                'min': activations.min().item()
            }

        results[method] = {
            'final_loss': losses[-1],
            'test_loss': test_loss.item(),
            'class_ious': ious,
            'mean_minority_iou': np.mean(ious[1:]),  # Classes 1,2,3
            'background_iou': ious[0],
            'train_time': train_time,
            'activation_stats': activation_stats,
            'loss_curve': losses
        }

        print(f"Final train loss: {losses[-1]:.4f}")
        print(f"Test loss: {test_loss.item():.4f}")
        print(f"Class IoUs: {[f'{iou:.3f}' for iou in ious]}")
        print(f"Mean minority IoU: {np.mean(ious[1:]):.3f}")
        print(f"Training time: {train_time:.2f}s")

    return results

def analyze_results(results):
    """Analyze and compare results"""
    print("\n" + "="*80)
    print("COMPREHENSIVE COMPARISON")
    print("="*80)

    methods = list(results.keys())

    # Performance comparison
    print(f"{'Method':<15} {'Train Loss':<12} {'Test Loss':<12} {'Bg IoU':<10} {'Min IoU':<10} {'Time':<8}")
    print("-" * 75)

    for method in methods:
        r = results[method]
        print(f"{method:<15} {r['final_loss']:<12.4f} {r['test_loss']:<12.4f} "
              f"{r['background_iou']:<10.3f} {r['mean_minority_iou']:<10.3f} {r['train_time']:<8.1f}s")

    # Activation analysis
    print(f"\n{'Method':<15} {'Act Mean':<10} {'Act Std':<10} {'Sparsity':<10} {'NonZero':<10} {'Range':<10}")
    print("-" * 75)

    for method in methods:
        stats = results[method]['activation_stats']
        range_val = stats['max'] - stats['min']
        print(f"{method:<15} {stats['mean']:<10.4f} {stats['std']:<10.4f} "
              f"{stats['sparsity']:<10.3f} {stats['nonzero_ratio']:<10.3f} {range_val:<10.3f}")

    # Key insights
    print(f"\nKEY INSIGHTS:")
    best_minority = max(methods, key=lambda m: results[m]['mean_minority_iou'])
    fastest = min(methods, key=lambda m: results[m]['train_time'])
    most_stable = min(methods, key=lambda m: results[m]['test_loss'])

    print(f"Best for minority classes: {best_minority.upper()} (IoU: {results[best_minority]['mean_minority_iou']:.3f})")
    print(f"Fastest training: {fastest.upper()} ({results[fastest]['train_time']:.1f}s)")
    print(f"Most stable: {most_stable.upper()} (test loss: {results[most_stable]['test_loss']:.4f})")

    # Check if your method is actually useful
    your_method_minority = results['enhanced']['mean_minority_iou']
    baseline_minority = results['standard']['mean_minority_iou']

    print(f"\nYOUR METHOD VERDICT:")
    if your_method_minority > baseline_minority * 1.1:  # 10% improvement
        print(f"✅ USEFUL! {(your_method_minority/baseline_minority-1)*100:.1f}% improvement on minority classes")
    elif your_method_minority > baseline_minority * 0.9:  # Within 10%
        print(f"🤔 MARGINAL. Only {(your_method_minority/baseline_minority-1)*100:.1f}% change on minority classes")
    else:
        print(f"❌ NOT HELPFUL. {(1-your_method_minority/baseline_minority)*100:.1f}% worse on minority classes")

if __name__ == "__main__":
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)

    print("Testing activation methods on sparse segmentation task...")
    print("Input: 4 channels, 85% sparsity")
    print("Labels: 4 classes, 97-99% background")

    results = test_activation_methods()
    analyze_results(results)

Testing activation methods on sparse segmentation task...
Input: 4 channels, 85% sparsity
Labels: 4 classes, 97-99% background
Using device: cpu
Input sparsity: 0.851
Background ratio: 0.967

=== Testing STANDARD ===
Epoch 0: Loss 1.4535, Grad norm: 7.0075
Epoch 5: Loss 0.9962, Grad norm: 5.6418
Epoch 10: Loss 0.6763, Grad norm: 4.4197
Epoch 15: Loss 0.4488, Grad norm: 3.0516
Final train loss: 0.3343
Test loss: 0.2575
Class IoUs: ['0.988', '0.000', '0.000', '0.000']
Mean minority IoU: 0.000
Training time: 3.04s

=== Testing GROUPMINMAX ===
Epoch 0: Loss 1.3999, Grad norm: 2.8505
Epoch 5: Loss 1.2318, Grad norm: 3.2626
Epoch 10: Loss 1.0085, Grad norm: 4.0585
Epoch 15: Loss 0.7220, Grad norm: 4.4799
Final train loss: 0.4857
Test loss: 0.4166
Class IoUs: ['0.988', '0.000', '0.000', '0.000']
Mean minority IoU: 0.000
Training time: 5.42s

=== Testing ENHANCED ===
Epoch 0: Loss 1.3828, Grad norm: 2.1406
Epoch 5: Loss 1.2928, Grad norm: 2.2658
Epoch 10: Loss 1.1849, Grad norm: 2.5599
Epoch 1

/tmp/ipython-input-1278125241.py:302: RuntimeWarning: invalid value encountered in scalar divide
  print(f"❌ NOT HELPFUL. {(1-your_method_minority/baseline_minority)*100:.1f}% worse on minority classes")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class VectorPreservingActivation(nn.Module):
    """
    Activation that preserves vector directions while normalizing magnitudes

    Key principle: Each spatial location is a vector in channel space
    - Preserve direction (relative channel relationships)
    - Normalize magnitude (for stable gradients)
    - Amplify/suppress based on vector characteristics, not individual channels
    """
    def __init__(self, channels, layer_depth='early', eps=1e-8):
        super().__init__()
        self.channels = channels
        self.layer_depth = layer_depth
        self.eps = eps

        # Layer-specific parameters
        if layer_depth == 'early':
            # Preserve input vectors strongly
            self.residual_strength = 0.8
            self.magnitude_amplification = 1.0
            self.direction_sensitivity = 0.1
        elif layer_depth == 'middle':
            # Allow more transformation while preserving key directions
            self.residual_strength = 0.6
            self.magnitude_amplification = 1.2
            self.direction_sensitivity = 0.3
        else:  # 'late'
            # Focus on class-specific vector patterns
            self.residual_strength = 0.7
            self.magnitude_amplification = 1.1
            self.direction_sensitivity = 0.5

        # Learnable class-specific direction templates (for imbalanced classes)
        # These learn the "0.4x, 1.3y, 0.33z" patterns you mentioned
        self.class_templates = nn.Parameter(torch.randn(4, channels) * 0.1)  # 4 classes
        self.template_sensitivity = nn.Parameter(torch.tensor(0.1))

    def vector_wise_normalization(self, x):
        """
        Normalize each spatial vector while preserving direction

        x: [B, C, H, W] where each (h,w) location is a C-dimensional vector
        """
        B, C, H, W = x.shape

        # Reshape to treat each spatial location as a vector
        x_vectors = x.permute(0, 2, 3, 1)  # [B, H, W, C]
        x_flat = x_vectors.reshape(B * H * W, C)  # [B*H*W, C]

        # Compute vector magnitudes (L2 norm)
        vector_magnitudes = torch.norm(x_flat, dim=1, keepdim=True)  # [B*H*W, 1]

        # Avoid division by zero for sparse vectors
        safe_magnitudes = vector_magnitudes.clamp(min=self.eps)

        # Normalize to unit vectors (preserve direction)
        unit_vectors = x_flat / safe_magnitudes  # [B*H*W, C]

        # Apply learned magnitude scaling based on original magnitude distribution
        magnitude_scale = self.compute_adaptive_magnitude_scale(vector_magnitudes)

        # Reconstruct with new magnitudes
        normalized_vectors = unit_vectors * magnitude_scale

        # Reshape back
        normalized = normalized_vectors.view(B, H, W, C).permute(0, 3, 1, 2)

        return normalized, vector_magnitudes.view(B, H, W, 1).permute(0, 3, 1, 2)

    def compute_adaptive_magnitude_scale(self, original_magnitudes):
        """
        Compute adaptive magnitude scaling that:
        - Amplifies weak but non-zero vectors (potential minority class signals)
        - Normalizes strong vectors to prevent explosion
        - Maintains sparsity for truly zero vectors
        """
        # Compute statistics over non-zero vectors only
        non_zero_mask = original_magnitudes > self.eps

        if non_zero_mask.sum() > 0:
            non_zero_mags = original_magnitudes[non_zero_mask]
            median_mag = torch.median(non_zero_mags)

            # Adaptive scaling: amplify weak signals, normalize strong ones
            scale_factor = torch.where(
                original_magnitudes < median_mag,
                self.magnitude_amplification * (median_mag / (original_magnitudes + self.eps)),
                median_mag / (original_magnitudes + self.eps)
            )

            # Keep zero vectors as zero
            scale_factor = torch.where(non_zero_mask, scale_factor, 0.0)
        else:
            scale_factor = torch.ones_like(original_magnitudes)

        return scale_factor

    def class_template_matching(self, x):
        """
        Amplify vectors that match learned class templates

        This learns the "0.4x, 1.3y, 0.33z" patterns for each class
        """
        B, C, H, W = x.shape

        # Reshape for template matching
        x_vectors = x.permute(0, 2, 3, 1).reshape(B * H * W, C)  # [B*H*W, C]

        # Compute similarity to each class template
        # templates: [4, C], x_vectors: [B*H*W, C]
        similarities = torch.mm(x_vectors, self.class_templates.t())  # [B*H*W, 4]

        # For each vector, find best matching template
        best_match_scores, best_templates = torch.max(similarities, dim=1)  # [B*H*W]

        # Amplify vectors that strongly match any class template
        amplification = 1.0 + self.template_sensitivity * torch.sigmoid(best_match_scores)
        amplification = amplification.unsqueeze(1)  # [B*H*W, 1]

        # Apply amplification
        amplified_vectors = x_vectors * amplification

        # Reshape back
        amplified = amplified_vectors.view(B, H, W, C).permute(0, 3, 1, 2)

        return amplified

    def direction_preserving_activation(self, x):
        """
        Apply activation that preserves vector directions

        Instead of channel-wise activation, apply activation to magnitude
        while keeping direction intact
        """
        B, C, H, W = x.shape

        # Get vector directions and magnitudes
        x_vectors = x.permute(0, 2, 3, 1).reshape(B * H * W, C)
        magnitudes = torch.norm(x_vectors, dim=1, keepdim=True).clamp(min=self.eps)
        directions = x_vectors / magnitudes

        # Apply activation to magnitudes only
        activated_magnitudes = magnitudes * torch.sigmoid(magnitudes)

        # Reconstruct vectors with activated magnitudes but preserved directions
        activated_vectors = directions * activated_magnitudes

        # Reshape back
        activated = activated_vectors.view(B, H, W, C).permute(0, 3, 1, 2)

        return activated

    def forward(self, x):
        original_x = x

        # Step 1: Vector-wise normalization (preserve directions)
        normalized, original_magnitudes = self.vector_wise_normalization(x)

        # Step 2: Class template matching (amplify class-relevant vectors)
        if self.layer_depth in ['middle', 'late']:
            template_matched = self.class_template_matching(normalized)
        else:
            template_matched = normalized

        # Step 3: Direction-preserving activation
        activated = self.direction_preserving_activation(template_matched)

        # Step 4: Strong residual to preserve original sparse structure
        output = self.residual_strength * original_x + (1 - self.residual_strength) * activated

        return output


class SparsityAwareVectorNorm(nn.Module):
    """
    Alternative approach: Normalization that specifically handles sparse vectors
    """
    def __init__(self, channels, sparsity_threshold=0.01):
        super().__init__()
        self.channels = channels
        self.sparsity_threshold = sparsity_threshold

        # Learnable parameters for sparse vs dense vector handling
        self.sparse_scale = nn.Parameter(torch.tensor(1.0))
        self.dense_scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        B, C, H, W = x.shape

        # Compute vector magnitudes
        x_vectors = x.permute(0, 2, 3, 1).reshape(B * H * W, C)
        magnitudes = torch.norm(x_vectors, dim=1, keepdim=True)

        # Classify vectors as sparse or dense
        sparse_mask = magnitudes < self.sparsity_threshold

        # Different normalization for sparse vs dense vectors
        directions = x_vectors / (magnitudes + 1e-8)

        # Sparse vectors: gentle scaling to preserve weak signals
        # Dense vectors: stronger normalization
        scales = torch.where(
            sparse_mask,
            self.sparse_scale * magnitudes,  # Preserve weak signals
            self.dense_scale / (magnitudes + 1e-8)  # Normalize strong signals
        )

        normalized_vectors = directions * scales
        return normalized_vectors.view(B, H, W, C).permute(0, 3, 1, 2)


# Test the vector preservation concept
def test_vector_preservation():
    """Test that our activation preserves vector directions"""

    # Create test vectors with known directions
    B, C, H, W = 2, 4, 8, 8

    # Class 1: vectors like [0.4, 1.3, 0.33, 0.1]
    class1_template = torch.tensor([0.4, 1.3, 0.33, 0.1])

    # Class 2: vectors like [0.8, 0.2, 0.9, 0.4]
    class2_template = torch.tensor([0.8, 0.2, 0.9, 0.4])

    # Create input with these patterns + noise + sparsity
    x = torch.zeros(B, C, H, W)

    for b in range(B):
        for h in range(H):
            for w in range(W):
                if torch.rand(1) < 0.15:  # 15% non-sparse (85% sparse)
                    if torch.rand(1) < 0.1:  # 10% class 1 (minority)
                        scale = torch.rand(1) * 2 + 0.5  # Random scale
                        noise = torch.randn(4) * 0.1  # Small noise
                        x[b, :, h, w] = scale * class1_template + noise
                    elif torch.rand(1) < 0.1:  # 10% class 2 (minority)
                        scale = torch.rand(1) * 2 + 0.5
                        noise = torch.randn(4) * 0.1
                        x[b, :, h, w] = scale * class2_template + noise
                    else:  # Background/noise
                        x[b, :, h, w] = torch.randn(4) * 0.3

    print(f"Input sparsity: {(torch.norm(x, dim=1) < 0.01).float().mean():.3f}")

    # Test vector-preserving activation
    activation = VectorPreservingActivation(C, 'middle')
    output = activation(x)

    print(f"Output sparsity: {(torch.norm(output, dim=1) < 0.01).float().mean():.3f}")

    # Check direction preservation for non-sparse vectors
    input_norms = torch.norm(x, dim=1)
    output_norms = torch.norm(output, dim=1)

    non_sparse_mask = input_norms > 0.01

    if non_sparse_mask.sum() > 0:
        # Compute cosine similarity between input and output vectors
        x_flat = x.permute(0, 2, 3, 1)[non_sparse_mask]
        out_flat = output.permute(0, 2, 3, 1)[non_sparse_mask]

        cosine_sim = F.cosine_similarity(x_flat, out_flat, dim=1)
        avg_cosine_sim = cosine_sim.mean().item()

        print(f"Average direction preservation (cosine similarity): {avg_cosine_sim:.3f}")
        print(f"Direction preservation score: {'GOOD' if avg_cosine_sim > 0.8 else 'POOR'}")


if __name__ == "__main__":
    print("Vector Direction Preserving Activation Test")
    print("="*50)
    test_vector_preservation()

    print("\nKey Insights:")
    print("• Each spatial location = vector in channel space")
    print("• Preserve vector direction = preserve relative channel relationships")
    print("• Normalize magnitude = stable gradients")
    print("• Learn class templates = generalized '0.4x, 1.3y, 0.33z' patterns")
    print("• Strong residual = preserve original sparse structure")

Vector Direction Preserving Activation Test
Input sparsity: 0.859
Output sparsity: 0.859
Average direction preservation (cosine similarity): 1.000
Direction preservation score: GOOD

Key Insights:
• Each spatial location = vector in channel space
• Preserve vector direction = preserve relative channel relationships
• Normalize magnitude = stable gradients
• Learn class templates = generalized '0.4x, 1.3y, 0.33z' patterns
• Strong residual = preserve original sparse structure


In [ ]:
class HybridVectorActivation(nn.Module):
    def __init__(self, channels, num_classes=4, reduction_ratio=8, eps=1e-6):
        super().__init__()
        self.eps = eps

        # Vector normalization core
        self.magnitude_norm = nn.InstanceNorm2d(1, affine=False)  # For magnitudes only

        # Your template matching
        self.class_templates = nn.Parameter(torch.randn(num_classes, channels) * 0.1)
        self.template_scale = nn.Parameter(torch.tensor(0.1))

        # My feature gating
        self.vector_gate = nn.Sequential(
            nn.Conv2d(channels, channels//reduction_ratio, 1),
            nn.GELU(),
            nn.Conv2d(channels//reduction_ratio, 1, 1),
            nn.Tanh()
        )

        # Your depth-aware settings (simplified)
        self.residual_weight = nn.Parameter(torch.tensor(0.2))

    def forward(self, x):
        zero_mask = (x == 0)

        # Direction preservation
        magnitudes = torch.norm(x, p=2, dim=1, keepdim=True).clamp(min=self.eps)
        unit_vectors = x / magnitudes

        # Your template matching (enhanced)
        B, C, H, W = x.shape
        x_flat = unit_vectors.permute(0,2,3,1).reshape(-1, C)
        similarities = F.cosine_similarity(
            x_flat.unsqueeze(1),
            self.class_templates.unsqueeze(0),
            dim=2
        )
        template_boost = 1 + self.template_scale * similarities.max(dim=1)[0]
        template_boost = template_boost.view(B, H, W, 1).permute(0,3,1,2)

        # My magnitude scaling
        magnitude_scale = self.vector_gate(x) + 1
        magnitude_scale = self.magnitude_norm(magnitude_scale) * template_boost

        # Reconstruct and residual
        activated = unit_vectors * magnitudes * magnitude_scale
        return torch.where(zero_mask, 0,
            self.residual_weight*x + (1-self.residual_weight)*activated
        )

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class VectorScaler(nn.Module):
    """
    Non-learnable vector normalization and activation module that preserves
    directional relationships while scaling magnitudes for computational stability
    and signal enhancement.

    Philosophy:
    - Normalization: Pure geometric scaling preserving all relative relationships
    - Activation: Magnitude-based filtering without directional changes
    - Both operations are deterministic and non-learnable
    """

    def __init__(self,
                 norm_mode: str = 'minmax',      # 'minmax', 'zscore', 'unit', 'absmax'
                 act_mode: str = 'threshold',    # 'threshold', 'sigmoid', 'relu', 'none'
                 act_threshold: float = 0.1,
                 spatial_dims: int = 3,          # Number of spatial dimensions (H, W, D)
                 sparse_aware: bool = True,      # Use sparse-aware normalization
                 eps: float = 1e-9):
        super().__init__()
        self.norm_mode = norm_mode
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.spatial_dims = spatial_dims
        self.sparse_aware = sparse_aware
        self.eps = eps

    def forward(self, x):
        """
        Input: (batch, channels, *spatial_dims) tensor
               e.g., (B, C, H, W, D) where each spatial location is a C-dimensional vector
        Output: Normalized and activated vectors preserving:
                - Exact zero positions
                - Vector directions in channel space
                - Relative spatial relationships
        """
        original_shape = x.shape
        batch_size, channels = x.shape[:2]
        spatial_shape = x.shape[2:]

        # Reshape to (batch, channels, num_spatial_locations)
        x_flat = x.view(batch_size, channels, -1)
        # Transpose to (batch, num_spatial_locations, channels) for vector operations
        x_vectors = x_flat.transpose(1, 2)  # Shape: (B, N_spatial, C)

        # 1. Identify exact zero vectors (all channels = 0)
        zero_mask = (x_vectors == 0).all(dim=-1, keepdim=True)  # Shape: (B, N_spatial, 1)

        # 2. Compute vector magnitudes (L2 norm across channel dimension)
        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True).clamp(min=self.eps)

        # 3. Extract unit directions
        directions = x_vectors / magnitudes

        # 4. Normalize magnitudes (non-learnable scaling)
        norm_magnitudes = self._normalize(magnitudes)

        # 5. Apply magnitude-only activation
        if self.act_mode != 'none':
            activated_magnitudes = self._activate(norm_magnitudes)
        else:
            activated_magnitudes = norm_magnitudes

        # 6. Reconstruct vectors maintaining exact directions
        result_vectors = directions * activated_magnitudes

        # 7. Restore exact zeros
        result_vectors = torch.where(zero_mask, torch.zeros_like(x_vectors), result_vectors)

        # 8. Reshape back to original format
        result_flat = result_vectors.transpose(1, 2)  # Back to (B, C, N_spatial)
        result = result_flat.view(original_shape)

        return result

    def _normalize(self, magnitudes):
        """
        Non-learnable magnitude normalization that preserves relative relationships

        Args:
            magnitudes: (B, N_spatial, 1) tensor of vector magnitudes
        Returns:
            normalized_magnitudes: Same shape, scaled magnitudes
        """
        if self.norm_mode == 'minmax':
            if self.sparse_aware:
                # Only consider non-zero magnitudes for statistics
                non_zero_mask = magnitudes > self.eps

                if non_zero_mask.any():
                    # Compute statistics only from non-zero magnitudes
                    non_zero_mags = magnitudes[non_zero_mask]
                    min_val = non_zero_mags.min()
                    max_val = non_zero_mags.max()
                    range_val = (max_val - min_val).clamp(min=self.eps)

                    # Apply normalization
                    normalized = (magnitudes - min_val) / range_val
                    # Ensure non-negative (zeros stay at or near zero)
                    normalized = torch.clamp(normalized, min=0.0, max=1.0)
                else:
                    normalized = magnitudes
            else:
                # Standard minmax across entire tensor
                min_val = magnitudes.amin(dim=(0, 1), keepdim=True)
                max_val = magnitudes.amax(dim=(0, 1), keepdim=True)
                range_val = (max_val - min_val).clamp(min=self.eps)
                normalized = (magnitudes - min_val) / range_val

            return normalized

        elif self.norm_mode == 'zscore':
            if self.sparse_aware:
                # Compute statistics only from non-zero magnitudes
                non_zero_mask = magnitudes > self.eps

                if non_zero_mask.any():
                    non_zero_mags = magnitudes[non_zero_mask]
                    mean_val = non_zero_mags.mean()
                    std_val = non_zero_mags.std().clamp(min=self.eps)
                    normalized = (magnitudes - mean_val) / std_val
                else:
                    normalized = magnitudes
            else:
                # Standard z-score normalization
                mean_val = magnitudes.mean(dim=(0, 1), keepdim=True)
                std_val = magnitudes.std(dim=(0, 1), keepdim=True).clamp(min=self.eps)
                normalized = (magnitudes - mean_val) / std_val

            return normalized

        elif self.norm_mode == 'unit':
            # Scale all non-zero magnitudes to unit length
            # This essentially makes all vectors unit vectors
            return torch.where(magnitudes > self.eps,
                             torch.ones_like(magnitudes),
                             magnitudes)

        elif self.norm_mode == 'absmax':
            # Scale by absolute maximum, preserving relative relationships
            if self.sparse_aware:
                non_zero_mask = magnitudes > self.eps
                if non_zero_mask.any():
                    max_val = magnitudes[non_zero_mask].max()
                else:
                    max_val = torch.tensor(1.0, device=magnitudes.device)
            else:
                max_val = magnitudes.amax()

            max_val = max_val.clamp(min=self.eps)
            return magnitudes / max_val

        else:
            raise ValueError(f"Unknown normalization mode: {self.norm_mode}")

    def _activate(self, magnitudes):
        """
        Non-learnable magnitude activation that amplifies or suppresses
        based on magnitude thresholds without changing directions

        Args:
            magnitudes: (B, N_spatial, 1) tensor of normalized magnitudes
        Returns:
            activated_magnitudes: Same shape, filtered magnitudes
        """
        if self.act_mode == 'threshold':
            # Piecewise linear activation
            return torch.where(
                magnitudes < self.act_threshold,
                magnitudes * 0.2,  # Suppress weak signals
                magnitudes * 2.0   # Amplify strong signals
            )

        elif self.act_mode == 'sigmoid':
            # Smooth sigmoid-based modulation
            # Scale magnitudes by sigmoid of their relative strength
            sigmoid_factor = torch.sigmoid((magnitudes - self.act_threshold) / self.act_threshold)
            return magnitudes * (0.2 + 1.8 * sigmoid_factor)  # Range: 0.2x to 2.0x

        elif self.act_mode == 'relu':
            # ReLU-style threshold with residual
            threshold_component = F.relu(magnitudes - self.act_threshold)
            return self.act_threshold * 0.1 + threshold_component  # Small baseline + thresholded

        elif self.act_mode == 'soft_threshold':
            # Soft thresholding (shrinkage)
            return torch.sign(magnitudes) * F.relu(torch.abs(magnitudes) - self.act_threshold)

        else:
            raise ValueError(f"Unknown activation mode: {self.act_mode}")

    def get_stats(self, x):
        """
        Utility method to analyze the statistics of input tensors
        Useful for debugging and understanding your data distribution
        """
        with torch.no_grad():
            original_shape = x.shape
            x_flat = x.view(x.shape[0], x.shape[1], -1).transpose(1, 2)

            # Compute magnitudes
            magnitudes = torch.norm(x_flat, p=2, dim=-1)
            zero_vectors = (x_flat == 0).all(dim=-1)

            stats = {
                'input_shape': original_shape,
                'total_vectors': magnitudes.numel(),
                'zero_vectors': zero_vectors.sum().item(),
                'sparsity_ratio': zero_vectors.float().mean().item(),
                'magnitude_stats': {
                    'min': magnitudes[magnitudes > 0].min().item() if (magnitudes > 0).any() else 0,
                    'max': magnitudes.max().item(),
                    'mean': magnitudes[magnitudes > 0].mean().item() if (magnitudes > 0).any() else 0,
                    'std': magnitudes[magnitudes > 0].std().item() if (magnitudes > 0).any() else 0,
                }
            }

        return stats


# Example usage and testing
if __name__ == "__main__":
    # Test with sparse 3D data similar to your use case
    batch_size, channels, H, W, D = 2, 16, 128, 128, 128

    # Create sparse test data (85% zeros)
    x = torch.randn(batch_size, channels, H, W, D)
    sparse_mask = torch.rand_like(x) < 0.85  # 85% sparsity
    x[sparse_mask] = 0

    # Initialize scaler
    scaler = VectorScaler(
        norm_mode='minmax',
        act_mode='threshold',
        act_threshold=0.1,
        sparse_aware=True
    )

    # Get input statistics
    input_stats = scaler.get_stats(x)
    print("Input Statistics:")
    for key, value in input_stats.items():
        print(f"  {key}: {value}")

    # Apply scaling
    output = scaler(x)

    # Get output statistics
    output_stats = scaler.get_stats(output)
    print("\nOutput Statistics:")
    for key, value in output_stats.items():
        print(f"  {key}: {value}")

    # Verify zero preservation
    input_zeros = (x == 0).all(dim=1, keepdim=True)
    output_zeros = (output == 0).all(dim=1, keepdim=True)
    zeros_preserved = torch.equal(input_zeros, output_zeros)
    print(f"\nZeros preserved: {zeros_preserved}")

In [ ]:
####################

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class VectorScaler(nn.Module):
    """
    Non-learnable vector normalization and activation module that preserves
    directional relationships while scaling magnitudes for computational stability
    and signal enhancement.

    Philosophy:
    - Normalization: Pure geometric scaling preserving all relative relationships
    - Activation: Magnitude-based filtering without directional changes
    - Both operations are deterministic and non-learnable
    """

    def __init__(self,
                 norm_mode: str = 'minmax',      # 'minmax', 'zscore', 'unit', 'absmax'
                 act_mode: str = 'threshold',    # 'threshold', 'sigmoid', 'relu', 'none'
                 act_threshold: float = 0.1,
                 spatial_dims: int = 3,          # Number of spatial dimensions (H, W, D)
                 sparse_aware: bool = True,      # Use sparse-aware normalization
                 eps: float = 1e-9):
        super().__init__()
        self.norm_mode = norm_mode
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.spatial_dims = spatial_dims
        self.sparse_aware = sparse_aware
        self.eps = eps

    def forward(self, x):
        """
        Input: (batch, channels, *spatial_dims) tensor
               e.g., (B, C, H, W, D) where each spatial location is a C-dimensional vector
        Output: Normalized and activated vectors preserving:
                - Exact zero positions
                - Vector directions in channel space
                - Relative spatial relationships
        """
        original_shape = x.shape
        batch_size, channels = x.shape[:2]
        spatial_shape = x.shape[2:]

        # Reshape to (batch, channels, num_spatial_locations)
        x_flat = x.view(batch_size, channels, -1)
        # Transpose to (batch, num_spatial_locations, channels) for vector operations
        x_vectors = x_flat.transpose(1, 2)  # Shape: (B, N_spatial, C)

        # 1. Identify exact zero vectors (all channels = 0)
        zero_mask = (x_vectors == 0).all(dim=-1, keepdim=True)  # Shape: (B, N_spatial, 1)

        # 2. Compute vector magnitudes (L2 norm across channel dimension)
        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True).clamp(min=self.eps)

        # 3. Extract unit directions
        directions = x_vectors / magnitudes

        # 4. Normalize magnitudes (non-learnable scaling)
        norm_magnitudes = self._normalize(magnitudes)

        # 5. Apply magnitude-only activation
        if self.act_mode != 'none':
            activated_magnitudes = self._activate(norm_magnitudes)
        else:
            activated_magnitudes = norm_magnitudes

        # 6. Reconstruct vectors maintaining exact directions
        result_vectors = directions * activated_magnitudes

        # 7. Restore exact zeros
        result_vectors = torch.where(zero_mask, torch.zeros_like(x_vectors), result_vectors)

        # 8. Reshape back to original format
        result_flat = result_vectors.transpose(1, 2)  # Back to (B, C, N_spatial)
        result = result_flat.view(original_shape)

        return result

    def _normalize(self, magnitudes):
        """
        Non-learnable magnitude normalization that preserves relative relationships

        Args:
            magnitudes: (B, N_spatial, 1) tensor of vector magnitudes
        Returns:
            normalized_magnitudes: Same shape, scaled magnitudes
        """
        if self.norm_mode == 'minmax':
            if self.sparse_aware:
                # Only consider non-zero magnitudes for statistics
                non_zero_mask = magnitudes > self.eps

                if non_zero_mask.any():
                    # Compute statistics only from non-zero magnitudes
                    non_zero_mags = magnitudes[non_zero_mask]
                    min_val = non_zero_mags.min()
                    max_val = non_zero_mags.max()
                    range_val = (max_val - min_val).clamp(min=self.eps)

                    # Apply normalization
                    normalized = (magnitudes - min_val) / range_val
                    # Ensure non-negative (zeros stay at or near zero)
                    normalized = torch.clamp(normalized, min=0.0, max=1.0)
                else:
                    normalized = magnitudes
            else:
                # Standard minmax across entire tensor
                min_val = magnitudes.amin(dim=(0, 1), keepdim=True)
                max_val = magnitudes.amax(dim=(0, 1), keepdim=True)
                range_val = (max_val - min_val).clamp(min=self.eps)
                normalized = (magnitudes - min_val) / range_val

            return normalized

        elif self.norm_mode == 'zscore':
            if self.sparse_aware:
                # Compute statistics only from non-zero magnitudes
                non_zero_mask = magnitudes > self.eps

                if non_zero_mask.any():
                    non_zero_mags = magnitudes[non_zero_mask]
                    mean_val = non_zero_mags.mean()
                    std_val = non_zero_mags.std().clamp(min=self.eps)
                    normalized = (magnitudes - mean_val) / std_val
                else:
                    normalized = magnitudes
            else:
                # Standard z-score normalization
                mean_val = magnitudes.mean(dim=(0, 1), keepdim=True)
                std_val = magnitudes.std(dim=(0, 1), keepdim=True).clamp(min=self.eps)
                normalized = (magnitudes - mean_val) / std_val

            return normalized

        elif self.norm_mode == 'unit':
            # Scale all non-zero magnitudes to unit length
            # This essentially makes all vectors unit vectors
            return torch.where(magnitudes > self.eps,
                             torch.ones_like(magnitudes),
                             magnitudes)

        elif self.norm_mode == 'absmax':
            # Scale by absolute maximum, preserving relative relationships
            if self.sparse_aware:
                non_zero_mask = magnitudes > self.eps
                if non_zero_mask.any():
                    max_val = magnitudes[non_zero_mask].max()
                else:
                    max_val = torch.tensor(1.0, device=magnitudes.device)
            else:
                max_val = magnitudes.amax()

            max_val = max_val.clamp(min=self.eps)
            return magnitudes / max_val

        else:
            raise ValueError(f"Unknown normalization mode: {self.norm_mode}")

    def _activate(self, magnitudes):
        """
        Non-learnable magnitude activation that amplifies or suppresses
        based on magnitude thresholds without changing directions

        Args:
            magnitudes: (B, N_spatial, 1) tensor of normalized magnitudes
        Returns:
            activated_magnitudes: Same shape, filtered magnitudes
        """
        if self.act_mode == 'threshold':
            # Piecewise linear activation
            return torch.where(
                magnitudes < self.act_threshold,
                magnitudes * 0.2,  # Suppress weak signals
                magnitudes * 2.0   # Amplify strong signals
            )

        elif self.act_mode == 'sigmoid':
            # Smooth sigmoid-based modulation
            # Scale magnitudes by sigmoid of their relative strength
            sigmoid_factor = torch.sigmoid((magnitudes - self.act_threshold) / self.act_threshold)
            return magnitudes * (0.2 + 1.8 * sigmoid_factor)  # Range: 0.2x to 2.0x

        elif self.act_mode == 'relu':
            # ReLU-style threshold with residual
            threshold_component = F.relu(magnitudes - self.act_threshold)
            return self.act_threshold * 0.1 + threshold_component  # Small baseline + thresholded

        elif self.act_mode == 'soft_threshold':
            # Soft thresholding (shrinkage)
            return torch.sign(magnitudes) * F.relu(torch.abs(magnitudes) - self.act_threshold)

        else:
            raise ValueError(f"Unknown activation mode: {self.act_mode}")

    def get_stats(self, x):
        """
        Utility method to analyze the statistics of input tensors
        Useful for debugging and understanding your data distribution
        """
        with torch.no_grad():
            original_shape = x.shape
            x_flat = x.view(x.shape[0], x.shape[1], -1).transpose(1, 2)

            # Compute magnitudes
            magnitudes = torch.norm(x_flat, p=2, dim=-1)
            zero_vectors = (x_flat == 0).all(dim=-1)

            stats = {
                'input_shape': original_shape,
                'total_vectors': magnitudes.numel(),
                'zero_vectors': zero_vectors.sum().item(),
                'sparsity_ratio': zero_vectors.float().mean().item(),
                'magnitude_stats': {
                    'min': magnitudes[magnitudes > 0].min().item() if (magnitudes > 0).any() else 0,
                    'max': magnitudes.max().item(),
                    'mean': magnitudes[magnitudes > 0].mean().item() if (magnitudes > 0).any() else 0,
                    'std': magnitudes[magnitudes > 0].std().item() if (magnitudes > 0).any() else 0,
                }
            }

        return stats


# Example usage and testing
if __name__ == "__main__":
    # Test with sparse 3D data similar to your use case
    batch_size, channels, H, W, D = 2, 16, 128, 128, 128

    # Create sparse test data (85% zeros)
    x = torch.randn(batch_size, channels, H, W, D)
    sparse_mask = torch.rand_like(x) < 0.85  # 85% sparsity
    x[sparse_mask] = 0

    # Initialize scaler
    scaler = VectorScaler(
        norm_mode='minmax',
        act_mode='threshold',
        act_threshold=0.1,
        sparse_aware=True
    )

    # Get input statistics
    input_stats = scaler.get_stats(x)
    print("Input Statistics:")
    for key, value in input_stats.items():
        print(f"  {key}: {value}")

    # Apply scaling
    output = scaler(x)

    # Get output statistics
    output_stats = scaler.get_stats(output)
    print("\nOutput Statistics:")
    for key, value in output_stats.items():
        print(f"  {key}: {value}")

    # Verify zero preservation
    input_zeros = (x == 0).all(dim=1, keepdim=True)
    output_zeros = (output == 0).all(dim=1, keepdim=True)
    zeros_preserved = torch.equal(input_zeros, output_zeros)
    print(f"\nZeros preserved: {zeros_preserved}")

In [ ]:
class VectorScaler(nn.Module):
    """
    non-learnable vector processor for sparse, high-dimensional data.
    Preserves geometric relationships while normalizing magnitudes and applying
    magnitude-only activation.

    Features:
    - Exact zero preservation (critical for sparse data)
    - Direction preservation in channel space
    - Multiple normalization strategies
    - Configurable activation thresholds
    - Sparse-aware statistics computation
    - Automatic shape handling (2D/3D/ND)
    - Built-in statistics reporting
    """

    def __init__(self,
                 norm_mode: str = 'minmax',      # 'minmax'|'zscore'|'unit'|'absmax'
                 act_mode: str = 'threshold',    # 'threshold'|'sigmoid'|'relu'|'none'
                 act_threshold: float = 0.1,
                 sparse_aware: bool = True,
                 eps: float = 1e-9):
        super().__init__()
        self.norm_mode = norm_mode
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.sparse_aware = sparse_aware
        self.eps = eps

    def forward(self, x):
        # Preserve input shape
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        # Flatten spatial dimensions
        x_flat = x.view(batch_size, channels, -1)  # [B, C, N]
        x_vectors = x_flat.permute(0, 2, 1)       # [B, N, C]

        # 1. Identify zero vectors (all channels = 0)
        zero_mask = (x_vectors == 0).all(dim=-1, keepdim=True)

        # 2. Compute magnitudes and directions
        with torch.no_grad():
            magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
            safe_magnitudes = magnitudes.clamp(min=self.eps)
            directions = x_vectors / safe_magnitudes

        # 3. Normalize magnitudes
        norm_mags = self._normalize(magnitudes)

        # 4. Apply activation if needed
        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        # 5. Reconstruct vectors
        result = directions * activated_mags

        # 6. Restore exact zeros and original shape
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize(self, m):
        """Core normalization logic with sparse awareness"""
        if self.sparse_aware:
            non_zero = m > self.eps
            if not non_zero.any():
                return m

            active_mags = m[non_zero]

            if self.norm_mode == 'minmax':
                min_val = active_mags.min()
                max_val = active_mags.max()
                norm = (m - min_val) / (max_val - min_val).clamp(min=self.eps)
                return torch.clamp(norm, 0, 1)

            elif self.norm_mode == 'zscore':
                mean = active_mags.mean()
                std = active_mags.std().clamp(min=self.eps)
                return (m - mean) / std

            elif self.norm_mode == 'unit':
                return m / m.clamp(min=self.eps)

            elif self.norm_mode == 'absmax':
                scale = active_mags.max().clamp(min=self.eps)
                return m / scale
        else:
            # Dense normalization variants
            if self.norm_mode == 'minmax':
                min_val = m.min()
                max_val = m.max()
                return (m - min_val) / (max_val - min_val).clamp(min=self.eps)

            elif self.norm_mode == 'zscore':
                return (m - m.mean()) / m.std().clamp(min=self.eps)

            elif self.norm_mode == 'unit':
                return m / m.clamp(min=self.eps)

            elif self.norm_mode == 'absmax':
                return m / m.max().clamp(min=self.eps)

        raise ValueError(f"Invalid norm_mode: {self.norm_mode}")

    def _activate(self, m):
        """Magnitude-only activation functions"""
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold,
                             m * 0.2,
                             m * 2.0)

        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold)/self.act_threshold)
            return m * (0.2 + 1.8 * scale)

        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.1)

        elif self.act_mode == 'soft_threshold':
            return F.softshrink(m, self.act_threshold)

        raise ValueError(f"Invalid act_mode: {self.act_mode}")

    def analyze(self, x):
        """Comprehensive input/output analysis"""
        with torch.no_grad():
            # Input stats
            flat = x.view(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
            mag = torch.norm(flat, dim=-1)
            zeros = (flat == 0).all(dim=-1)

            stats = {
                'shape': x.shape,
                'sparsity': zeros.float().mean().item(),
                'magnitude': {
                    'min': mag[mag > 0].min().item() if (mag > 0).any() else 0,
                    'max': mag.max().item(),
                    'mean': mag[mag > 0].mean().item() if (mag > 0).any() else 0,
                    'std': mag[mag > 0].std().item() if (mag > 0).any() else 0,
                }
            }
        return stats

In [ ]:
class VectorScaler(nn.Module):
    """
    non-learnable vector processor for sparse, high-dimensional data.
    Preserves geometric relationships while normalizing magnitudes and applying
    magnitude-only activation.

    Features:
    - Exact zero preservation (critical for sparse data)
    - Direction preservation in channel space
    - Multiple normalization strategies
    - Configurable activation thresholds
    - Sparse-aware statistics computation
    - Automatic shape handling (2D/3D/ND)
    - Built-in statistics reporting
    """

    def __init__(self,
                 norm_mode: str = 'minmax',      # 'minmax'|'zscore'|'unit'|'absmax'
                 act_mode: str = 'threshold',    # 'threshold'|'sigmoid'|'relu'|'none'
                 act_threshold: float = 0.1,
                 sparse_aware: bool = True,
                 eps: float = 1e-9):
        super().__init__()
        self.norm_mode = norm_mode
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.sparse_aware = sparse_aware
        self.eps = eps

    def forward(self, x):
        # Preserve input shape
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        # Flatten spatial dimensions
        x_flat = x.view(batch_size, channels, -1)  # [B, C, N]
        x_vectors = x_flat.permute(0, 2, 1)       # [B, N, C]

        # 1. Identify zero vectors (all channels = 0)
        zero_mask = (x_vectors == 0).all(dim=-1, keepdim=True)

        # 2. Compute magnitudes and directions
        with torch.no_grad():
            magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
            safe_magnitudes = magnitudes.clamp(min=self.eps)
            directions = x_vectors / safe_magnitudes

        # 3. Normalize magnitudes
        norm_mags = self._normalize(magnitudes)

        # 4. Apply activation if needed
        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        # 5. Reconstruct vectors
        result = directions * activated_mags

        # 6. Restore exact zeros and original shape
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize(self, m):
        """Core normalization logic with sparse awareness"""
        if self.sparse_aware:
            non_zero = m > self.eps
            if not non_zero.any():
                return m

            active_mags = m[non_zero]

            self.norm_mode == 'minmax':
            min_val = active_mags.min()
            max_val = active_mags.max()
            norm = (m - min_val) / (max_val - min_val).clamp(min=self.eps)
            return torch.clamp(norm, 0, 1)

    def _activate(self, m):
        """Magnitude-only activation functions"""
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold,
                             m * 0.2,
                             m * 2.0)

        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold)/self.act_threshold)
            return m * (0.2 + 1.8 * scale)

        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.1)

        elif self.act_mode == 'soft_threshold':
            return F.softshrink(m, self.act_threshold)

        raise ValueError(f"Invalid act_mode: {self.act_mode}")

    def analyze(self, x):
        """Comprehensive input/output analysis"""
        with torch.no_grad():
            # Input stats
            flat = x.view(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
            mag = torch.norm(flat, dim=-1)
            zeros = (flat == 0).all(dim=-1)

            stats = {
                'shape': x.shape,
                'sparsity': zeros.float().mean().item(),
                'magnitude': {
                    'min': mag[mag > 0].min().item() if (mag > 0).any() else 0,
                    'max': mag.max().item(),
                    'mean': mag[mag > 0].mean().item() if (mag > 0).any() else 0,
                    'std': mag[mag > 0].std().item() if (mag > 0).any() else 0,
                }
            }
        return stats

In [ ]:
x = torch.randn(size=(1, 256, 10, 15, 15))
norm = VectorScaler()
print(x.shape)
x = norm(x)
print(x.shape)

torch.Size([1, 256, 10, 15, 15])
torch.Size([1, 256, 10, 15, 15])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VectorScaler(nn.Module):
    """
    Streamlined vector processor for sparse data.
    Direction preservation + MinMax normalization + Magnitude-only activation.

    Key features:
    - Exact zero preservation
    - Direction preservation in channel space
    - MinMax normalization only (simple & fast)
    - Magnitude-only activation
    """

    def __init__(self,
                 act_mode: str = 'threshold',    # 'threshold'|'sigmoid'|'relu'|'none'
                 act_threshold: float = 0.1,
                 eps: float = 1e-9):
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps

    def forward(self, x):
        # Preserve input shape
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        # Flatten spatial dimensions
        x_flat = x.view(batch_size, channels, -1)  # [B, C, N]
        x_vectors = x_flat.permute(0, 2, 1)       # [B, N, C]

        # 1. Identify zero vectors (all channels = 0)
        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        # 2. Compute magnitudes and directions
        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        # Handle directions carefully for zero vectors
        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        # 3. MinMax normalize magnitudes (sparse-aware)
        norm_mags = self._normalize_minmax(magnitudes)

        # 4. Apply activation if needed
        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        # 5. Reconstruct vectors: direction × magnitude
        result = directions * activated_mags

        # 6. Restore exact zeros and original shape
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_minmax(self, magnitudes):
        """Simple MinMax normalization - sparse aware"""
        # Find non-zero magnitudes across entire batch
        non_zero_mask = magnitudes > self.eps

        if not non_zero_mask.any():
            return magnitudes  # All zeros, nothing to normalize

        # Get min/max from non-zero values only
        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        # Avoid division by zero
        range_val = (max_val - min_val).clamp(min=self.eps)

        # Normalize: (x - min) / (max - min)
        normalized = (magnitudes - min_val) / range_val

        # Clamp to [0, 1] range
        return torch.clamp(normalized, 0, 1)

    def _activate(self, m):
        """Magnitude-only activation functions"""
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold,
                             m * 0.2,      # Suppress weak signals
                             m * 1.5)      # Moderate amplification

        elif self.act_mode == 'sigmoid':
            # Smooth transition around threshold
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.2 + 1.3 * scale)  # Range: 0.2x to 1.5x

        elif self.act_mode == 'relu':
            # Hard threshold with small leak
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.1)

        raise ValueError(f"Invalid act_mode: {self.act_mode}")


# Quick test function
def test_vectorscaler():
    """Quick functionality test"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Create test data (small for speed)
    batch_size, channels, h, w = 2, 8, 32, 32
    x = torch.randn(batch_size, channels, h, w, device=device)

    # Make it sparse (85% zeros)
    sparse_mask = torch.rand_like(x) < 0.15
    x = x * sparse_mask.float()

    print(f"Input shape: {x.shape}")
    print(f"Input sparsity: {(x.abs() < 1e-9).float().mean().item():.3f}")
    print(f"Input magnitude range: {x.abs().max().item():.4f}")

    # Test scaler
    scaler = VectorScaler(act_mode='threshold', act_threshold=0.2).to(device)

    with torch.no_grad():
        y = scaler(x)

    print(f"Output sparsity: {(y.abs() < 1e-9).float().mean().item():.3f}")
    print(f"Output magnitude range: {y.abs().max().item():.4f}")
    print("✓ Test passed!")

if __name__ == "__main__":
    test_vectorscaler()

Input shape: torch.Size([2, 8, 32, 32])
Input sparsity: 0.844
Input magnitude range: 3.5424
Output sparsity: 0.844
Output magnitude range: 1.4550
✓ Test passed!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class VectorScaler(nn.Module):
    """
    Streamlined vector processor for sparse data.
    Direction preservation + MinMax normalization + Magnitude-only activation.
    """
    def __init__(self, act_mode: str = 'threshold', act_threshold: float = 0.1, eps: float = 1e-9):
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        norm_mags = self._normalize_minmax(magnitudes)

        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_minmax(self, magnitudes):
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()
        range_val = (max_val - min_val).clamp(min=self.eps)
        normalized = (magnitudes - min_val) / range_val
        return torch.clamp(normalized, 0, 1)

    def _activate(self, m):
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold, m * 0.2, m * 1.5)
        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.2 + 1.3 * scale)
        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.1)
        raise ValueError(f"Invalid act_mode: {self.act_mode}")


def test_direction_preservation():
    """Test 1: Direction Preservation - Core theoretical requirement"""
    print("🧭 TEST 1: Direction Preservation")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='none').to(device)  # No activation, pure normalization

    # Create vectors with known directions
    batch_size, channels = 4, 8
    x = torch.zeros(batch_size, channels, 10, 10, device=device)

    # Set specific directional vectors
    x[0, :, 5, 5] = torch.tensor([1, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Unit vector along channel 0
    x[1, :, 5, 5] = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Diagonal direction
    x[2, :, 5, 5] = torch.tensor([10, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype) # Same direction, different magnitude
    x[3, :, 5, 5] = torch.tensor([0, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Zero vector

    with torch.no_grad():
        y = scaler(x)

    # Check direction preservation using cosine similarity
    input_vec = x.view(batch_size, channels, -1)[:, :, 55]  # Get the [5,5] position vectors
    output_vec = y.view(batch_size, channels, -1)[:, :, 55]

    for i in range(batch_size):
        inp = input_vec[i]
        out = output_vec[i]

        if torch.norm(inp) < 1e-6:  # Zero vector case
            assert torch.norm(out) < 1e-6, f"Zero vector not preserved at sample {i}"
            print(f"  ✅ Sample {i}: Zero vector preserved")
        else:
            cos_sim = F.cosine_similarity(inp.unsqueeze(0), out.unsqueeze(0), dim=1)
            assert cos_sim > 0.999, f"Direction not preserved at sample {i}: cosine={cos_sim.item()}"
            print(f"  ✅ Sample {i}: Direction preserved (cosine={cos_sim.item():.6f})")

    print("✅ Direction preservation test PASSED!\n")


def test_extreme_sparsity():
    """Test 2: Extreme Sparsity (99%+ zeros) - Your real-world scenario"""
    print("🕸️  TEST 2: Extreme Sparsity (99%+ zeros)")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='threshold', act_threshold=0.3).to(device)

    # Create 99.5% sparse data
    x = torch.randn(2, 16, 64, 64, device=device)
    sparse_mask = torch.rand_like(x) < 0.005  # Keep only 0.5%
    x = x * sparse_mask.float()

    input_sparsity = (x.abs() < 1e-9).float().mean()
    print(f"  Input sparsity: {input_sparsity.item():.4f}")

    with torch.no_grad():
        y = scaler(x)

    output_sparsity = (y.abs() < 1e-9).float().mean()
    print(f"  Output sparsity: {output_sparsity.item():.4f}")

    # Check that zero structure is preserved
    input_zeros = (x.abs() < 1e-9)
    output_zeros = (y.abs() < 1e-9)
    zero_preservation = (input_zeros == output_zeros).float().mean()

    print(f"  Zero structure preserved: {zero_preservation.item():.6f}")
    assert zero_preservation > 0.999, "Zero structure not preserved!"
    print("✅ Extreme sparsity test PASSED!\n")


def test_magnitude_scaling():
    """Test 3: Magnitude Scaling Behavior - Verify activation functions"""
    print("📏 TEST 3: Magnitude Scaling Verification")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Test different activation modes
    modes = ['threshold', 'sigmoid', 'relu']

    for mode in modes:
        print(f"  Testing {mode} activation:")
        scaler = VectorScaler(act_mode=mode, act_threshold=0.4).to(device)

        # Create vectors with controlled magnitudes
        x = torch.zeros(3, 4, 10, 10, device=device)
        x[0, :, 5, 5] = torch.tensor([0.1, 0, 0, 0], device=device, dtype=x.dtype)  # Low magnitude
        x[1, :, 5, 5] = torch.tensor([0.5, 0, 0, 0], device=device, dtype=x.dtype)  # Medium magnitude
        x[2, :, 5, 5] = torch.tensor([2.0, 0, 0, 0], device=device, dtype=x.dtype)  # High magnitude

        with torch.no_grad():
            y = scaler(x)

        input_mags = torch.norm(x.view(3, 4, -1), dim=1)[:, 55]
        output_mags = torch.norm(y.view(3, 4, -1), dim=1)[:, 55]

        for i in range(3):
            mag_labels = ['Low', 'Med', 'High']
            print(f"    {mag_labels[i]} mag: {input_mags[i].item():.3f} → {output_mags[i].item():.3f}")

        print()

    print("✅ Magnitude scaling test PASSED!\n")


def test_batch_consistency():
    """Test 4: Batch Processing Consistency - Important for training"""
    print("🎯 TEST 4: Batch Consistency")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='threshold').to(device)

    # Create identical samples in a batch
    x_single = torch.randn(1, 8, 16, 16, device=device)
    x_single = x_single * (torch.rand_like(x_single) < 0.2)  # Make sparse

    # Process single sample
    with torch.no_grad():
        y_single = scaler(x_single)

    # Process same sample in batch of 4
    x_batch = x_single.repeat(4, 1, 1, 1)
    with torch.no_grad():
        y_batch = scaler(x_batch)

    # Check consistency across batch
    for i in range(4):
        diff = torch.abs(y_single[0] - y_batch[i]).max()
        print(f"  Sample {i} max difference: {diff.item():.2e}")
        assert diff < 1e-6, f"Batch inconsistency detected at sample {i}"

    print("✅ Batch consistency test PASSED!\n")


def test_gradient_flow():
    """Test 5: Gradient Flow - Critical for training"""
    print("🌊 TEST 5: Gradient Flow")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='sigmoid').to(device)

    # Create input requiring gradients
    x = torch.randn(2, 8, 16, 16, device=device, requires_grad=True)
    x.data = x.data * (torch.rand_like(x.data) < 0.3)  # Make sparse

    # Forward pass
    y = scaler(x)
    loss = y.sum()  # Simple loss

    # Backward pass
    loss.backward()

    # Check gradient statistics
    grad_exists = x.grad is not None
    if grad_exists:
        grad_norm = torch.norm(x.grad)
        grad_sparsity = (x.grad.abs() < 1e-9).float().mean()
        print(f"  Gradient exists: {grad_exists}")
        print(f"  Gradient norm: {grad_norm.item():.6f}")
        print(f"  Gradient sparsity: {grad_sparsity.item():.4f}")

        # Check for NaN or exploding gradients
        has_nan = torch.isnan(x.grad).any()
        has_inf = torch.isinf(x.grad).any()
        print(f"  Has NaN gradients: {has_nan.item()}")
        print(f"  Has Inf gradients: {has_inf.item()}")

        assert not has_nan, "NaN gradients detected!"
        assert not has_inf, "Infinite gradients detected!"
        assert grad_norm < 100, "Exploding gradients detected!"

    print("✅ Gradient flow test PASSED!\n")


def test_memory_efficiency():
    """Test 6: Memory Efficiency - No unnecessary computations"""
    print("💾 TEST 6: Memory Efficiency")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler().to(device)

    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    # Process reasonably large tensor
    x = torch.randn(4, 32, 128, 128, device=device)
    x = x * (torch.rand_like(x) < 0.1)  # 90% sparse

    if device.type == 'cuda':
        mem_before = torch.cuda.memory_allocated()

    with torch.no_grad():
        y = scaler(x)

    if device.type == 'cuda':
        mem_after = torch.cuda.memory_allocated()
        mem_peak = torch.cuda.max_memory_allocated()

        print(f"  Memory before: {mem_before / 1024**2:.1f} MB")
        print(f"  Memory after: {mem_after / 1024**2:.1f} MB")
        print(f"  Peak memory: {mem_peak / 1024**2:.1f} MB")
        print(f"  Input tensor size: {x.numel() * x.element_size() / 1024**2:.1f} MB")

        # Memory should be reasonable (not more than 5x input size during processing)
        reasonable_peak = mem_before + 5 * (x.numel() * x.element_size())
        assert mem_peak < reasonable_peak, "Excessive memory usage!"
    else:
        print("  CPU mode - memory tracking limited")

    print("✅ Memory efficiency test PASSED!\n")


def test_edge_cases():
    """Test 7: Edge Cases - Robustness verification"""
    print("⚠️  TEST 7: Edge Cases")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler().to(device)

    edge_cases = [
        ("All zeros", torch.zeros(2, 4, 8, 8, device=device)),
        ("All same value", torch.ones(2, 4, 8, 8, device=device) * 0.5),
        ("Single non-zero", torch.zeros(2, 4, 8, 8, device=device)),
        ("Very small values", torch.ones(2, 4, 8, 8, device=device) * 1e-8),
        ("Very large values", torch.ones(2, 4, 8, 8, device=device) * 1e6),
    ]

    # Set single non-zero for that case
    edge_cases[2][1][0, 0, 4, 4] = 1.0

    for case_name, x in edge_cases:
        print(f"  Testing: {case_name}")
        try:
            with torch.no_grad():
                y = scaler(x)

            # Check for NaN or Inf
            has_nan = torch.isnan(y).any()
            has_inf = torch.isinf(y).any()

            print(f"    Output range: [{y.min().item():.2e}, {y.max().item():.2e}]")
            print(f"    Has NaN: {has_nan.item()}, Has Inf: {has_inf.item()}")

            assert not has_nan, f"NaN detected in {case_name}"
            assert not has_inf, f"Inf detected in {case_name}"
            print(f"    ✅ {case_name} handled correctly")

        except Exception as e:
            print(f"    ❌ {case_name} failed: {e}")
            raise

        print()

    print("✅ Edge cases test PASSED!\n")


def run_comprehensive_tests():
    """Run all test suites"""
    print("🧪 COMPREHENSIVE VECTORSCALER TEST SUITE")
    print("=" * 70)
    print()

    tests = [
        test_direction_preservation,
        test_extreme_sparsity,
        test_magnitude_scaling,
        test_batch_consistency,
        test_gradient_flow,
        test_memory_efficiency,
        test_edge_cases
    ]

    passed = 0
    failed = 0

    for test in tests:
        try:
            test()
            passed += 1
        except Exception as e:
            print(f"❌ TEST FAILED: {e}")
            failed += 1
            print()

    print("=" * 70)
    print(f"📊 TEST SUMMARY: {passed} PASSED, {failed} FAILED")

    if failed == 0:
        print("🎉 ALL TESTS PASSED! Your VectorScaler is robust and ready for production!")
    else:
        print("⚠️  Some tests failed. Please review the issues above.")

    print("=" * 70)


if __name__ == "__main__":
    run_comprehensive_tests()

🧪 COMPREHENSIVE VECTORSCALER TEST SUITE

🧭 TEST 1: Direction Preservation
❌ TEST FAILED: Direction not preserved at sample 0: cosine=0.0

🕸️  TEST 2: Extreme Sparsity (99%+ zeros)
  Input sparsity: 0.9950
  Output sparsity: 0.9950
  Zero structure preserved: 0.999992
✅ Extreme sparsity test PASSED!

📏 TEST 3: Magnitude Scaling Verification
  Testing threshold activation:
    Low mag: 0.100 → 0.000
    Med mag: 0.500 → 0.042
    High mag: 2.000 → 1.500

  Testing sigmoid activation:
    Low mag: 0.100 → 0.000
    Med mag: 0.500 → 0.147
    High mag: 2.000 → 1.263

  Testing relu activation:
    Low mag: 0.100 → 0.040
    Med mag: 0.500 → 0.040
    High mag: 2.000 → 0.640

✅ Magnitude scaling test PASSED!

🎯 TEST 4: Batch Consistency
  Sample 0 max difference: 0.00e+00
  Sample 1 max difference: 0.00e+00
  Sample 2 max difference: 0.00e+00
  Sample 3 max difference: 0.00e+00
✅ Batch consistency test PASSED!

🌊 TEST 5: Gradient Flow
  Gradient exists: True
  Gradient norm: 22.533970
  Gra

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class VectorScaler(nn.Module):
    """
    Streamlined vector processor for sparse data.
    Direction preservation + MinMax normalization + Magnitude-only activation.
    """
    def __init__(self, act_mode: str = 'threshold', act_threshold: float = 0.1, eps: float = 1e-9):
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        norm_mags = self._normalize_minmax(magnitudes)

        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_minmax(self, magnitudes):
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()
        range_val = (max_val - min_val).clamp(min=self.eps)
        normalized = (magnitudes - min_val) / range_val
        return torch.clamp(normalized, 0, 1)

    def _activate(self, m):
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold, m * 0.2, m * 1.5)
        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.2 + 1.3 * scale)
        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.1)
        raise ValueError(f"Invalid act_mode: {self.act_mode}")


def test_direction_preservation():
    """Test 1: Direction Preservation - Core theoretical requirement"""
    print("🧭 TEST 1: Direction Preservation")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='none').to(device)  # No activation, pure normalization

    # Create vectors with known directions - need multiple non-zero vectors for proper normalization
    batch_size, channels = 4, 8
    x = torch.zeros(batch_size, channels, 10, 10, device=device)

    # Set specific directional vectors - ADD MORE NON-ZERO VECTORS for proper min-max normalization
    x[0, :, 5, 5] = torch.tensor([1, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Unit vector along channel 0
    x[0, :, 6, 6] = torch.tensor([2, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Same direction, different magnitude

    x[1, :, 5, 5] = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Diagonal direction
    x[1, :, 6, 6] = torch.tensor([3, 3, 3, 3, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Same direction, different magnitude

    x[2, :, 5, 5] = torch.tensor([0, 0, 0, 0, 1, 1, 0, 0], device=device, dtype=x.dtype) # Different direction
    x[2, :, 6, 6] = torch.tensor([0, 0, 0, 0, 5, 5, 0, 0], device=device, dtype=x.dtype) # Same direction, different magnitude

    x[3, :, 5, 5] = torch.tensor([0, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Zero vector

    with torch.no_grad():
        y = scaler(x)

    # Check direction preservation using cosine similarity
    # Test multiple positions for each sample
    test_positions = [55, 66]  # [5,5] and [6,6] positions in flattened tensor

    for i in range(batch_size-1):  # Skip last sample (all zeros)
        for pos_idx, pos in enumerate(test_positions):
            inp = x.view(batch_size, channels, -1)[i, :, pos]
            out = y.view(batch_size, channels, -1)[i, :, pos]

            if torch.norm(inp) < 1e-6:  # Skip zero vectors
                continue

            cos_sim = F.cosine_similarity(inp.unsqueeze(0), out.unsqueeze(0), dim=1)
            print(f"  Sample {i}, pos {pos_idx}: Direction preserved (cosine={cos_sim.item():.6f})")
            assert cos_sim > 0.999, f"Direction not preserved at sample {i}, pos {pos_idx}: cosine={cos_sim.item()}"

    # Test zero vector preservation
    zero_inp = x.view(batch_size, channels, -1)[3, :, 55]
    zero_out = y.view(batch_size, channels, -1)[3, :, 55]
    assert torch.norm(zero_out) < 1e-6, "Zero vector not preserved"
    print(f"  ✅ Sample 3: Zero vector preserved")

    print("✅ Direction preservation test PASSED!\n")


def test_extreme_sparsity():
    """Test 2: Real-world Sparsity (85-89% zeros) - Your actual data characteristics"""
    print("🕸️  TEST 2: Real-world Sparsity (85-89% zeros)")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='threshold', act_threshold=0.3).to(device)

    # Test different sparsity levels matching your data
    sparsity_levels = [0.85, 0.87, 0.89]

    for target_sparsity in sparsity_levels:
        # Create sparse data
        x = torch.randn(2, 16, 64, 64, device=device)
        keep_prob = 1.0 - target_sparsity
        sparse_mask = torch.rand_like(x) < keep_prob
        x = x * sparse_mask.float()

        input_sparsity = (x.abs() < 1e-9).float().mean()

        with torch.no_grad():
            y = scaler(x)

        output_sparsity = (y.abs() < 1e-9).float().mean()

        # Check that zero structure is preserved
        input_zeros = (x.abs() < 1e-9)
        output_zeros = (y.abs() < 1e-9)
        zero_preservation = (input_zeros == output_zeros).float().mean()

        print(f"  Target sparsity {target_sparsity:.2f}:")
        print(f"    Actual input sparsity: {input_sparsity.item():.4f}")
        print(f"    Output sparsity: {output_sparsity.item():.4f}")
        print(f"    Zero preservation: {zero_preservation.item():.6f}")

        assert zero_preservation > 0.999, f"Zero structure not preserved at {target_sparsity} sparsity!"
        assert abs(input_sparsity - output_sparsity) < 0.001, f"Sparsity changed at {target_sparsity}!"

    print("✅ Real-world sparsity test PASSED!\n")


def test_magnitude_scaling():
    """Test 3: Magnitude Scaling Behavior - Verify activation functions"""
    print("📏 TEST 3: Magnitude Scaling Verification")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Test different activation modes
    modes = ['threshold', 'sigmoid', 'relu']

    for mode in modes:
        print(f"  Testing {mode} activation:")
        scaler = VectorScaler(act_mode=mode, act_threshold=0.4).to(device)

        # Create vectors with controlled magnitudes
        x = torch.zeros(3, 4, 10, 10, device=device)
        x[0, :, 5, 5] = torch.tensor([0.1, 0, 0, 0], device=device, dtype=x.dtype)  # Low magnitude
        x[1, :, 5, 5] = torch.tensor([0.5, 0, 0, 0], device=device, dtype=x.dtype)  # Medium magnitude
        x[2, :, 5, 5] = torch.tensor([2.0, 0, 0, 0], device=device, dtype=x.dtype)  # High magnitude

        with torch.no_grad():
            y = scaler(x)

        input_mags = torch.norm(x.view(3, 4, -1), dim=1)[:, 55]
        output_mags = torch.norm(y.view(3, 4, -1), dim=1)[:, 55]

        for i in range(3):
            mag_labels = ['Low', 'Med', 'High']
            print(f"    {mag_labels[i]} mag: {input_mags[i].item():.3f} → {output_mags[i].item():.3f}")

        print()

    print("✅ Magnitude scaling test PASSED!\n")


def test_batch_consistency():
    """Test 4: Batch Processing Consistency - Important for training"""
    print("🎯 TEST 4: Batch Consistency")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='threshold').to(device)

    # Create identical samples in a batch
    x_single = torch.randn(1, 8, 16, 16, device=device)
    x_single = x_single * (torch.rand_like(x_single) < 0.2)  # Make sparse

    # Process single sample
    with torch.no_grad():
        y_single = scaler(x_single)

    # Process same sample in batch of 4
    x_batch = x_single.repeat(4, 1, 1, 1)
    with torch.no_grad():
        y_batch = scaler(x_batch)

    # Check consistency across batch
    for i in range(4):
        diff = torch.abs(y_single[0] - y_batch[i]).max()
        print(f"  Sample {i} max difference: {diff.item():.2e}")
        assert diff < 1e-6, f"Batch inconsistency detected at sample {i}"

    print("✅ Batch consistency test PASSED!\n")


def test_gradient_flow():
    """Test 5: Gradient Flow - Critical for training"""
    print("🌊 TEST 5: Gradient Flow")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='sigmoid').to(device)

    # Create input requiring gradients
    x = torch.randn(2, 8, 16, 16, device=device, requires_grad=True)
    x.data = x.data * (torch.rand_like(x.data) < 0.3)  # Make sparse

    # Forward pass
    y = scaler(x)
    loss = y.sum()  # Simple loss

    # Backward pass
    loss.backward()

    # Check gradient statistics
    grad_exists = x.grad is not None
    if grad_exists:
        grad_norm = torch.norm(x.grad)
        grad_sparsity = (x.grad.abs() < 1e-9).float().mean()
        print(f"  Gradient exists: {grad_exists}")
        print(f"  Gradient norm: {grad_norm.item():.6f}")
        print(f"  Gradient sparsity: {grad_sparsity.item():.4f}")

        # Check for NaN or exploding gradients
        has_nan = torch.isnan(x.grad).any()
        has_inf = torch.isinf(x.grad).any()
        print(f"  Has NaN gradients: {has_nan.item()}")
        print(f"  Has Inf gradients: {has_inf.item()}")

        assert not has_nan, "NaN gradients detected!"
        assert not has_inf, "Infinite gradients detected!"
        assert grad_norm < 100, "Exploding gradients detected!"

    print("✅ Gradient flow test PASSED!\n")


def test_memory_efficiency():
    """Test 6: Memory Efficiency - No unnecessary computations"""
    print("💾 TEST 6: Memory Efficiency")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler().to(device)

    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    # Process reasonably large tensor
    x = torch.randn(4, 32, 128, 128, device=device)
    x = x * (torch.rand_like(x) < 0.1)  # 90% sparse

    if device.type == 'cuda':
        mem_before = torch.cuda.memory_allocated()

    with torch.no_grad():
        y = scaler(x)

    if device.type == 'cuda':
        mem_after = torch.cuda.memory_allocated()
        mem_peak = torch.cuda.max_memory_allocated()

        print(f"  Memory before: {mem_before / 1024**2:.1f} MB")
        print(f"  Memory after: {mem_after / 1024**2:.1f} MB")
        print(f"  Peak memory: {mem_peak / 1024**2:.1f} MB")
        print(f"  Input tensor size: {x.numel() * x.element_size() / 1024**2:.1f} MB")

        # Memory should be reasonable (not more than 5x input size during processing)
        reasonable_peak = mem_before + 5 * (x.numel() * x.element_size())
        assert mem_peak < reasonable_peak, "Excessive memory usage!"
    else:
        print("  CPU mode - memory tracking limited")

    print("✅ Memory efficiency test PASSED!\n")


def test_edge_cases():
    """Test 7: Edge Cases - Robustness verification"""
    print("⚠️  TEST 7: Edge Cases")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler().to(device)

    edge_cases = [
        ("All zeros", torch.zeros(2, 4, 8, 8, device=device)),
        ("All same value", torch.ones(2, 4, 8, 8, device=device) * 0.5),
        ("Single non-zero", torch.zeros(2, 4, 8, 8, device=device)),
        ("Very small values", torch.ones(2, 4, 8, 8, device=device) * 1e-8),
        ("Very large values", torch.ones(2, 4, 8, 8, device=device) * 1e6),
    ]

    # Set single non-zero for that case
    edge_cases[2][1][0, 0, 4, 4] = 1.0

    for case_name, x in edge_cases:
        print(f"  Testing: {case_name}")
        try:
            with torch.no_grad():
                y = scaler(x)

            # Check for NaN or Inf
            has_nan = torch.isnan(y).any()
            has_inf = torch.isinf(y).any()

            print(f"    Output range: [{y.min().item():.2e}, {y.max().item():.2e}]")
            print(f"    Has NaN: {has_nan.item()}, Has Inf: {has_inf.item()}")

            assert not has_nan, f"NaN detected in {case_name}"
            assert not has_inf, f"Inf detected in {case_name}"
            print(f"    ✅ {case_name} handled correctly")

        except Exception as e:
            print(f"    ❌ {case_name} failed: {e}")
            raise

        print()

    print("✅ Edge cases test PASSED!\n")


def run_comprehensive_tests():
    """Run all test suites"""
    print("🧪 COMPREHENSIVE VECTORSCALER TEST SUITE")
    print("=" * 70)
    print()

    tests = [
        test_direction_preservation,
        test_extreme_sparsity,
        test_magnitude_scaling,
        test_batch_consistency,
        test_gradient_flow,
        test_memory_efficiency,
        test_edge_cases
    ]

    passed = 0
    failed = 0

    for test in tests:
        try:
            test()
            passed += 1
        except Exception as e:
            print(f"❌ TEST FAILED: {e}")
            failed += 1
            print()

    print("=" * 70)
    print(f"📊 TEST SUMMARY: {passed} PASSED, {failed} FAILED")

    if failed == 0:
        print("🎉 ALL TESTS PASSED! Your VectorScaler is robust and ready for production!")
    else:
        print("⚠️  Some tests failed. Please review the issues above.")

    print("=" * 70)


if __name__ == "__main__":
    run_comprehensive_tests()

🧪 COMPREHENSIVE VECTORSCALER TEST SUITE

🧭 TEST 1: Direction Preservation
  Sample 0, pos 0: Direction preserved (cosine=0.000000)
❌ TEST FAILED: Direction not preserved at sample 0, pos 0: cosine=0.0

🕸️  TEST 2: Real-world Sparsity (85-89% zeros)
  Target sparsity 0.85:
    Actual input sparsity: 0.8495
    Output sparsity: 0.8495
    Zero preservation: 0.999992
  Target sparsity 0.87:
    Actual input sparsity: 0.8701
    Output sparsity: 0.8701
    Zero preservation: 0.999992
  Target sparsity 0.89:
    Actual input sparsity: 0.8900
    Output sparsity: 0.8900
    Zero preservation: 0.999992
✅ Real-world sparsity test PASSED!

📏 TEST 3: Magnitude Scaling Verification
  Testing threshold activation:
    Low mag: 0.100 → 0.000
    Med mag: 0.500 → 0.042
    High mag: 2.000 → 1.500

  Testing sigmoid activation:
    Low mag: 0.100 → 0.000
    Med mag: 0.500 → 0.147
    High mag: 2.000 → 1.263

  Testing relu activation:
    Low mag: 0.100 → 0.040
    Med mag: 0.500 → 0.040
    High ma

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VectorScaler(nn.Module):
    """
    FIXED: Direction-preserving vector processor for sparse data.
    The key fix: normalization that doesn't map any magnitude to exactly zero.
    """

    def __init__(self,
                 act_mode: str = 'threshold',
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):  # NEW: Prevent zero magnitudes
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude  # Minimum allowed normalized magnitude

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        # 1. Identify zero vectors (all channels = 0)
        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        # 2. Compute magnitudes and directions
        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        # 3. FIXED: Direction-preserving normalization
        norm_mags = self._normalize_preserving(magnitudes)

        # 4. Apply activation if needed
        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        # 5. Reconstruct vectors: direction × magnitude
        result = directions * activated_mags

        # 6. Restore exact zeros and original shape
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        """
        FIXED: Normalization that preserves directions by ensuring no magnitude becomes exactly 0
        Maps magnitudes to range [min_magnitude, 1.0] instead of [0.0, 1.0]
        """
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            # All magnitudes are the same - return them as-is but scaled
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        # Map to [min_magnitude, 1.0] instead of [0.0, 1.0]
        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val  # This gives [0, 1]

        # Rescale to [min_magnitude, 1.0] so nothing becomes zero
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        """Magnitude-only activation functions"""
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)  # Less aggressive
        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.5 + 0.7 * scale)  # Range: 0.5x to 1.2x
        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.2)
        raise ValueError(f"Invalid act_mode: {self.act_mode}")


def test_fixed_direction_preservation():
    """Test the fix"""
    print("🧭 TESTING FIXED DIRECTION PRESERVATION")
    print("=" * 50)

    device = torch.device('cpu')
    scaler = VectorScaler(act_mode='none', min_magnitude=0.1).to(device)

    # Create test vectors
    x = torch.zeros(1, 4, 5, 5, device=device)
    x[0, :, 2, 2] = torch.tensor([1.0, 0.0, 0.0, 0.0])  # Will get min magnitude
    x[0, :, 2, 3] = torch.tensor([2.0, 0.0, 0.0, 0.0])  # Same direction, different magnitude
    x[0, :, 3, 2] = torch.tensor([0.0, 3.0, 0.0, 0.0])  # Different direction, max magnitude

    print("Input vectors:")
    positions = [(2, 2), (2, 3), (3, 2)]
    input_vecs = []
    for i, (r, c) in enumerate(positions):
        vec = x[0, :, r, c]
        input_vecs.append(vec)
        print(f"  Position {positions[i]}: {vec.tolist()}")

    with torch.no_grad():
        y = scaler(x)

    print("\nOutput vectors:")
    output_vecs = []
    for i, (r, c) in enumerate(positions):
        vec = y[0, :, r, c]
        output_vecs.append(vec)
        print(f"  Position {positions[i]}: {vec.tolist()}")

    print("\nDirection preservation:")
    for i, (inp, out) in enumerate(zip(input_vecs, output_vecs)):
        if torch.norm(inp) > 1e-6 and torch.norm(out) > 1e-6:
            cos_sim = F.cosine_similarity(inp.unsqueeze(0), out.unsqueeze(0), dim=1)
            print(f"  Position {positions[i]}: cosine = {cos_sim.item():.6f}")
            if cos_sim > 0.999:
                print(f"    ✅ Direction preserved!")
            else:
                print(f"    ❌ Direction NOT preserved!")
        else:
            print(f"  Position {positions[i]}: One vector is zero")


if __name__ == "__main__":
    test_fixed_direction_preservation()

🧭 TESTING FIXED DIRECTION PRESERVATION
Input vectors:
  Position (2, 2): [1.0, 0.0, 0.0, 0.0]
  Position (2, 3): [2.0, 0.0, 0.0, 0.0]
  Position (3, 2): [0.0, 3.0, 0.0, 0.0]

Output vectors:
  Position (2, 2): [0.10000000149011612, 0.0, 0.0, 0.0]
  Position (2, 3): [0.550000011920929, 0.0, 0.0, 0.0]
  Position (3, 2): [0.0, 1.0, 0.0, 0.0]

Direction preservation:
  Position (2, 2): cosine = 1.000000
    ✅ Direction preserved!
  Position (2, 3): cosine = 1.000000
    ✅ Direction preserved!
  Position (3, 2): cosine = 1.000000
    ✅ Direction preserved!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VectorScaler(nn.Module):
    """
    FIXED: Direction-preserving vector processor for sparse data.
    The key fix: normalization that doesn't map any magnitude to exactly zero.
    """

    def __init__(self,
                 act_mode: str = 'threshold',
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):  # NEW: Prevent zero magnitudes
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude  # Minimum allowed normalized magnitude

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        # 1. Identify zero vectors (all channels = 0)
        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        # 2. Compute magnitudes and directions
        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        # 3. FIXED: Direction-preserving normalization
        norm_mags = self._normalize_preserving(magnitudes)

        # 4. Apply activation if needed
        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        # 5. Reconstruct vectors: direction × magnitude
        result = directions * activated_mags

        # 6. Restore exact zeros and original shape
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        """
        FIXED: Normalization that preserves directions by ensuring no magnitude becomes exactly 0
        Maps magnitudes to range [min_magnitude, 1.0] instead of [0.0, 1.0]
        """
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            # All magnitudes are the same - return them as-is but scaled
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        # Map to [min_magnitude, 1.0] instead of [0.0, 1.0]
        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val  # This gives [0, 1]

        # Rescale to [min_magnitude, 1.0] so nothing becomes zero
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        """Magnitude-only activation functions"""
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)  # Less aggressive
        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.5 + 0.7 * scale)  # Range: 0.5x to 1.2x
        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.2)
        raise ValueError(f"Invalid act_mode: {self.act_mode}")


def test_direction_preservation():
    """Test 1: Direction Preservation - Core theoretical requirement"""
    print("🧭 TEST 1: Direction Preservation")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='none').to(device)  # No activation, pure normalization

    # Create vectors with known directions - need multiple non-zero vectors for proper normalization
    batch_size, channels = 4, 8
    x = torch.zeros(batch_size, channels, 10, 10, device=device)

    # Set specific directional vectors - ADD MORE NON-ZERO VECTORS for proper min-max normalization
    x[0, :, 5, 5] = torch.tensor([1, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Unit vector along channel 0
    x[0, :, 6, 6] = torch.tensor([2, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Same direction, different magnitude

    x[1, :, 5, 5] = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Diagonal direction
    x[1, :, 6, 6] = torch.tensor([3, 3, 3, 3, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Same direction, different magnitude

    x[2, :, 5, 5] = torch.tensor([0, 0, 0, 0, 1, 1, 0, 0], device=device, dtype=x.dtype) # Different direction
    x[2, :, 6, 6] = torch.tensor([0, 0, 0, 0, 5, 5, 0, 0], device=device, dtype=x.dtype) # Same direction, different magnitude

    x[3, :, 5, 5] = torch.tensor([0, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=x.dtype)  # Zero vector

    with torch.no_grad():
        y = scaler(x)

    # Check direction preservation using cosine similarity
    # Test multiple positions for each sample
    test_positions = [55, 66]  # [5,5] and [6,6] positions in flattened tensor

    for i in range(batch_size-1):  # Skip last sample (all zeros)
        for pos_idx, pos in enumerate(test_positions):
            inp = x.view(batch_size, channels, -1)[i, :, pos]
            out = y.view(batch_size, channels, -1)[i, :, pos]

            if torch.norm(inp) < 1e-6:  # Skip zero vectors
                continue

            cos_sim = F.cosine_similarity(inp.unsqueeze(0), out.unsqueeze(0), dim=1)
            print(f"  Sample {i}, pos {pos_idx}: Direction preserved (cosine={cos_sim.item():.6f})")
            assert cos_sim > 0.999, f"Direction not preserved at sample {i}, pos {pos_idx}: cosine={cos_sim.item()}"

    # Test zero vector preservation
    zero_inp = x.view(batch_size, channels, -1)[3, :, 55]
    zero_out = y.view(batch_size, channels, -1)[3, :, 55]
    assert torch.norm(zero_out) < 1e-6, "Zero vector not preserved"
    print(f"  ✅ Sample 3: Zero vector preserved")

    print("✅ Direction preservation test PASSED!\n")


def test_extreme_sparsity():
    """Test 2: Real-world Sparsity (85-89% zeros) - Your actual data characteristics"""
    print("🕸️  TEST 2: Real-world Sparsity (85-89% zeros)")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='threshold', act_threshold=0.3).to(device)

    # Test different sparsity levels matching your data
    sparsity_levels = [0.85, 0.87, 0.89]

    for target_sparsity in sparsity_levels:
        # Create sparse data
        x = torch.randn(2, 16, 64, 64, device=device)
        keep_prob = 1.0 - target_sparsity
        sparse_mask = torch.rand_like(x) < keep_prob
        x = x * sparse_mask.float()

        input_sparsity = (x.abs() < 1e-9).float().mean()

        with torch.no_grad():
            y = scaler(x)

        output_sparsity = (y.abs() < 1e-9).float().mean()

        # Check that zero structure is preserved
        input_zeros = (x.abs() < 1e-9)
        output_zeros = (y.abs() < 1e-9)
        zero_preservation = (input_zeros == output_zeros).float().mean()

        print(f"  Target sparsity {target_sparsity:.2f}:")
        print(f"    Actual input sparsity: {input_sparsity.item():.4f}")
        print(f"    Output sparsity: {output_sparsity.item():.4f}")
        print(f"    Zero preservation: {zero_preservation.item():.6f}")

        assert zero_preservation > 0.999, f"Zero structure not preserved at {target_sparsity} sparsity!"
        assert abs(input_sparsity - output_sparsity) < 0.001, f"Sparsity changed at {target_sparsity}!"

    print("✅ Real-world sparsity test PASSED!\n")


def test_magnitude_scaling():
    """Test 3: Magnitude Scaling Behavior - Verify activation functions"""
    print("📏 TEST 3: Magnitude Scaling Verification")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Test different activation modes
    modes = ['threshold', 'sigmoid', 'relu']

    for mode in modes:
        print(f"  Testing {mode} activation:")
        scaler = VectorScaler(act_mode=mode, act_threshold=0.4).to(device)

        # Create vectors with controlled magnitudes
        x = torch.zeros(3, 4, 10, 10, device=device)
        x[0, :, 5, 5] = torch.tensor([0.1, 0, 0, 0], device=device, dtype=x.dtype)  # Low magnitude
        x[1, :, 5, 5] = torch.tensor([0.5, 0, 0, 0], device=device, dtype=x.dtype)  # Medium magnitude
        x[2, :, 5, 5] = torch.tensor([2.0, 0, 0, 0], device=device, dtype=x.dtype)  # High magnitude

        with torch.no_grad():
            y = scaler(x)

        input_mags = torch.norm(x.view(3, 4, -1), dim=1)[:, 55]
        output_mags = torch.norm(y.view(3, 4, -1), dim=1)[:, 55]

        for i in range(3):
            mag_labels = ['Low', 'Med', 'High']
            print(f"    {mag_labels[i]} mag: {input_mags[i].item():.3f} → {output_mags[i].item():.3f}")

        print()

    print("✅ Magnitude scaling test PASSED!\n")


def test_batch_consistency():
    """Test 4: Batch Processing Consistency - Important for training"""
    print("🎯 TEST 4: Batch Consistency")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='threshold').to(device)

    # Create identical samples in a batch
    x_single = torch.randn(1, 8, 16, 16, device=device)
    x_single = x_single * (torch.rand_like(x_single) < 0.2)  # Make sparse

    # Process single sample
    with torch.no_grad():
        y_single = scaler(x_single)

    # Process same sample in batch of 4
    x_batch = x_single.repeat(4, 1, 1, 1)
    with torch.no_grad():
        y_batch = scaler(x_batch)

    # Check consistency across batch
    for i in range(4):
        diff = torch.abs(y_single[0] - y_batch[i]).max()
        print(f"  Sample {i} max difference: {diff.item():.2e}")
        assert diff < 1e-6, f"Batch inconsistency detected at sample {i}"

    print("✅ Batch consistency test PASSED!\n")


def test_gradient_flow():
    """Test 5: Gradient Flow - Critical for training"""
    print("🌊 TEST 5: Gradient Flow")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(act_mode='sigmoid').to(device)

    # Create input requiring gradients
    x = torch.randn(2, 8, 16, 16, device=device, requires_grad=True)
    x.data = x.data * (torch.rand_like(x.data) < 0.3)  # Make sparse

    # Forward pass
    y = scaler(x)
    loss = y.sum()  # Simple loss

    # Backward pass
    loss.backward()

    # Check gradient statistics
    grad_exists = x.grad is not None
    if grad_exists:
        grad_norm = torch.norm(x.grad)
        grad_sparsity = (x.grad.abs() < 1e-9).float().mean()
        print(f"  Gradient exists: {grad_exists}")
        print(f"  Gradient norm: {grad_norm.item():.6f}")
        print(f"  Gradient sparsity: {grad_sparsity.item():.4f}")

        # Check for NaN or exploding gradients
        has_nan = torch.isnan(x.grad).any()
        has_inf = torch.isinf(x.grad).any()
        print(f"  Has NaN gradients: {has_nan.item()}")
        print(f"  Has Inf gradients: {has_inf.item()}")

        assert not has_nan, "NaN gradients detected!"
        assert not has_inf, "Infinite gradients detected!"
        assert grad_norm < 100, "Exploding gradients detected!"

    print("✅ Gradient flow test PASSED!\n")


def test_memory_efficiency():
    """Test 6: Memory Efficiency - No unnecessary computations"""
    print("💾 TEST 6: Memory Efficiency")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler().to(device)

    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    # Process reasonably large tensor
    x = torch.randn(4, 32, 128, 128, device=device)
    x = x * (torch.rand_like(x) < 0.1)  # 90% sparse

    if device.type == 'cuda':
        mem_before = torch.cuda.memory_allocated()

    with torch.no_grad():
        y = scaler(x)

    if device.type == 'cuda':
        mem_after = torch.cuda.memory_allocated()
        mem_peak = torch.cuda.max_memory_allocated()

        print(f"  Memory before: {mem_before / 1024**2:.1f} MB")
        print(f"  Memory after: {mem_after / 1024**2:.1f} MB")
        print(f"  Peak memory: {mem_peak / 1024**2:.1f} MB")
        print(f"  Input tensor size: {x.numel() * x.element_size() / 1024**2:.1f} MB")

        # Memory should be reasonable (not more than 5x input size during processing)
        reasonable_peak = mem_before + 5 * (x.numel() * x.element_size())
        assert mem_peak < reasonable_peak, "Excessive memory usage!"
    else:
        print("  CPU mode - memory tracking limited")

    print("✅ Memory efficiency test PASSED!\n")


def test_edge_cases():
    """Test 7: Edge Cases - Robustness verification"""
    print("⚠️  TEST 7: Edge Cases")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler().to(device)

    edge_cases = [
        ("All zeros", torch.zeros(2, 4, 8, 8, device=device)),
        ("All same value", torch.ones(2, 4, 8, 8, device=device) * 0.5),
        ("Single non-zero", torch.zeros(2, 4, 8, 8, device=device)),
        ("Very small values", torch.ones(2, 4, 8, 8, device=device) * 1e-8),
        ("Very large values", torch.ones(2, 4, 8, 8, device=device) * 1e6),
    ]

    # Set single non-zero for that case
    edge_cases[2][1][0, 0, 4, 4] = 1.0

    for case_name, x in edge_cases:
        print(f"  Testing: {case_name}")
        try:
            with torch.no_grad():
                y = scaler(x)

            # Check for NaN or Inf
            has_nan = torch.isnan(y).any()
            has_inf = torch.isinf(y).any()

            print(f"    Output range: [{y.min().item():.2e}, {y.max().item():.2e}]")
            print(f"    Has NaN: {has_nan.item()}, Has Inf: {has_inf.item()}")

            assert not has_nan, f"NaN detected in {case_name}"
            assert not has_inf, f"Inf detected in {case_name}"
            print(f"    ✅ {case_name} handled correctly")

        except Exception as e:
            print(f"    ❌ {case_name} failed: {e}")
            raise

        print()

    print("✅ Edge cases test PASSED!\n")


def run_comprehensive_tests():
    """Run all test suites"""
    print("🧪 COMPREHENSIVE VECTORSCALER TEST SUITE")
    print("=" * 70)
    print()

    tests = [
        test_direction_preservation,
        test_extreme_sparsity,
        test_magnitude_scaling,
        test_batch_consistency,
        test_gradient_flow,
        test_memory_efficiency,
        test_edge_cases
    ]

    passed = 0
    failed = 0

    for test in tests:
        try:
            test()
            passed += 1
        except Exception as e:
            print(f"❌ TEST FAILED: {e}")
            failed += 1
            print()

    print("=" * 70)
    print(f"📊 TEST SUMMARY: {passed} PASSED, {failed} FAILED")

    if failed == 0:
        print("🎉 ALL TESTS PASSED! Your VectorScaler is robust and ready for production!")
    else:
        print("⚠️  Some tests failed. Please review the issues above.")

    print("=" * 70)


if __name__ == "__main__":
    run_comprehensive_tests()

🧪 COMPREHENSIVE VECTORSCALER TEST SUITE

🧭 TEST 1: Direction Preservation
  Sample 0, pos 0: Direction preserved (cosine=1.000000)
  Sample 0, pos 1: Direction preserved (cosine=1.000000)
  Sample 1, pos 0: Direction preserved (cosine=1.000000)
  Sample 1, pos 1: Direction preserved (cosine=1.000000)
  Sample 2, pos 0: Direction preserved (cosine=1.000000)
  Sample 2, pos 1: Direction preserved (cosine=1.000000)
  ✅ Sample 3: Zero vector preserved
✅ Direction preservation test PASSED!

🕸️  TEST 2: Real-world Sparsity (85-89% zeros)
  Target sparsity 0.85:
    Actual input sparsity: 0.8502
    Output sparsity: 0.8502
    Zero preservation: 1.000000
  Target sparsity 0.87:
    Actual input sparsity: 0.8702
    Output sparsity: 0.8702
    Zero preservation: 1.000000
  Target sparsity 0.89:
    Actual input sparsity: 0.8921
    Output sparsity: 0.8921
    Zero preservation: 1.000000
✅ Real-world sparsity test PASSED!

📏 TEST 3: Magnitude Scaling Verification
  Testing threshold activation:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VectorScaler(nn.Module):
    """Production-ready VectorScaler with fixed configuration"""

    def __init__(self,
                 act_mode: str = 'threshold',
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        norm_mags = self._normalize_preserving(magnitudes)

        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)
        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.5 + 0.7 * scale)
        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.2)
        raise ValueError(f"Invalid act_mode: {self.act_mode}")


def comprehensive_production_test():
    """
    ULTIMATE PRODUCTION TEST: Test all winning configurations against comprehensive test suite
    """
    print("🏭 ULTIMATE PRODUCTION VALIDATION TEST")
    print("=" * 80)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Testing on: {device}")

    # Your winning configurations
    winning_configs = [
        {'act_mode': 'threshold', 'act_threshold': 0.2, 'min_magnitude': 0.1, 'name': 'Config 1 (Recommended)'},
        {'act_mode': 'threshold', 'act_threshold': 0.3, 'min_magnitude': 0.15, 'name': 'Config 2 (Conservative)'},
        {'act_mode': 'sigmoid', 'act_threshold': 0.2, 'min_magnitude': 0.1, 'name': 'Config 3 (Smooth)'},
        {'act_mode': 'sigmoid', 'act_threshold': 0.25, 'min_magnitude': 0.1, 'name': 'Config 4 (Balanced)'},
    ]

    def run_test_suite(scaler, config_name):
        """Run comprehensive test suite for a given configuration"""
        results = {'passed': 0, 'failed': 0, 'details': []}

        tests = [
            ('Direction Preservation', test_direction_preservation),
            ('Real-world Sparsity', test_extreme_sparsity),
            ('Magnitude Scaling', test_magnitude_scaling),
            ('Batch Consistency', test_batch_consistency),
            ('Gradient Flow', test_gradient_flow),
            ('Memory Efficiency', test_memory_efficiency),
            ('Edge Cases', test_edge_cases),
            ('Production Stress', test_production_stress),
            ('Long Training Simulation', test_training_simulation)
        ]

        print(f"\n🧪 Testing {config_name}")
        print("-" * 60)

        for test_name, test_func in tests:
            try:
                test_func(scaler)
                results['passed'] += 1
                results['details'].append({'test': test_name, 'status': 'PASSED'})
                print(f"  ✅ {test_name}: PASSED")
            except Exception as e:
                results['failed'] += 1
                results['details'].append({'test': test_name, 'status': 'FAILED', 'error': str(e)})
                print(f"  ❌ {test_name}: FAILED - {str(e)[:100]}")

        total_tests = results['passed'] + results['failed']
        success_rate = results['passed'] / total_tests if total_tests > 0 else 0
        print(f"  📊 {config_name}: {results['passed']}/{total_tests} tests passed ({success_rate:.1%})")

        return results

    # Test all configurations
    all_results = {}
    for config in winning_configs:
        config_params = {k: v for k, v in config.items() if k != 'name'}
        scaler = VectorScaler(**config_params).to(device)
        all_results[config['name']] = run_test_suite(scaler, config['name'])

    # Final recommendation
    print(f"\n🏆 FINAL PRODUCTION RECOMMENDATION")
    print("=" * 80)

    best_config = None
    best_score = -1

    for config_name, results in all_results.items():
        total = results['passed'] + results['failed']
        score = results['passed'] / total if total > 0 else 0

        print(f"{config_name}:")
        print(f"  Success Rate: {score:.1%} ({results['passed']}/{total})")

        if results['failed'] > 0:
            failed_tests = [d['test'] for d in results['details'] if d['status'] == 'FAILED']
            print(f"  Failed Tests: {', '.join(failed_tests)}")

        if score > best_score:
            best_score = score
            best_config = config_name
        print()

    print(f"🚀 PRODUCTION WINNER: {best_config} ({best_score:.1%} success rate)")

    # Get the winning config parameters
    winner = next(c for c in winning_configs if c['name'] == best_config)
    winner_params = {k: v for k, v in winner.items() if k != 'name'}

    print(f"\n💼 COPY THIS FOR PRODUCTION:")
    print(f"scaler = VectorScaler(**{winner_params})")

    return winner_params


def test_direction_preservation(scaler):
    """Test direction preservation with multiple test cases"""
    device = next(scaler.parameters()).device

    # Test with multiple scenarios
    scenarios = [
        # Scenario 1: Clear directional vectors
        torch.zeros(4, 8, 10, 10, device=device),
        # Scenario 2: Random sparse vectors
        torch.randn(4, 8, 16, 16, device=device) * (torch.rand(4, 8, 16, 16, device=device) < 0.15)
    ]

    # Set up scenario 1
    scenarios[0][0, :, 5, 5] = torch.tensor([1, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=scenarios[0].dtype)
    scenarios[0][0, :, 6, 6] = torch.tensor([2, 0, 0, 0, 0, 0, 0, 0], device=device, dtype=scenarios[0].dtype)
    scenarios[0][1, :, 5, 5] = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0], device=device, dtype=scenarios[0].dtype)
    scenarios[0][1, :, 6, 6] = torch.tensor([3, 3, 3, 3, 0, 0, 0, 0], device=device, dtype=scenarios[0].dtype)

    for i, x in enumerate(scenarios):
        with torch.no_grad():
            y = scaler(x)

        # Sample non-zero vectors and check direction preservation
        x_flat = x.view(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        y_flat = y.view(y.shape[0], y.shape[1], -1).permute(0, 2, 1)

        non_zero_mask = torch.norm(x_flat, dim=-1) > 1e-6
        if non_zero_mask.sum() > 5:
            non_zero_indices = torch.nonzero(non_zero_mask)[:10]

            for idx in non_zero_indices:
                b, pos = idx[0], idx[1]
                vec_in = x_flat[b, pos]
                vec_out = y_flat[b, pos]

                if torch.norm(vec_out) > 1e-6:
                    cos_sim = F.cosine_similarity(vec_in.unsqueeze(0), vec_out.unsqueeze(0), dim=1)
                    if cos_sim < 0.95:
                        raise AssertionError(f"Direction not preserved: cosine={cos_sim.item():.4f}")


def test_extreme_sparsity(scaler):
    """Test with your actual sparsity levels"""
    device = next(scaler.parameters()).device

    sparsity_levels = [0.85, 0.87, 0.89, 0.91]  # Including even higher sparsity

    for sparsity in sparsity_levels:
        x = torch.randn(2, 16, 32, 32, device=device)
        keep_prob = 1.0 - sparsity
        x = x * (torch.rand_like(x) < keep_prob)

        input_sparsity = (x.abs() < 1e-9).float().mean()

        with torch.no_grad():
            y = scaler(x)

        output_sparsity = (y.abs() < 1e-9).float().mean()
        sparsity_diff = abs(input_sparsity - output_sparsity)

        if sparsity_diff > 0.01:
            raise AssertionError(f"Sparsity not preserved: {input_sparsity:.3f} vs {output_sparsity:.3f}")


def test_magnitude_scaling(scaler):
    """Test magnitude scaling behavior"""
    device = next(scaler.parameters()).device

    # Test different magnitude ranges
    x = torch.zeros(3, 4, 10, 10, device=device)
    x[0, :, 5, 5] = torch.tensor([0.05, 0, 0, 0], device=device, dtype=x.dtype)
    x[1, :, 5, 5] = torch.tensor([0.5, 0, 0, 0], device=device, dtype=x.dtype)
    x[2, :, 5, 5] = torch.tensor([2.0, 0, 0, 0], device=device, dtype=x.dtype)

    with torch.no_grad():
        y = scaler(x)

    # Just check for reasonable outputs (no NaN/Inf)
    if torch.isnan(y).any() or torch.isinf(y).any():
        raise AssertionError("NaN or Inf in magnitude scaling test")


def test_batch_consistency(scaler):
    """Test batch processing consistency"""
    device = next(scaler.parameters()).device

    x_single = torch.randn(1, 8, 16, 16, device=device)
    x_single = x_single * (torch.rand_like(x_single) < 0.2)

    with torch.no_grad():
        y_single = scaler(x_single)

        x_batch = x_single.repeat(4, 1, 1, 1)
        y_batch = scaler(x_batch)

    for i in range(4):
        diff = torch.abs(y_single[0] - y_batch[i]).max()
        if diff > 1e-5:
            raise AssertionError(f"Batch inconsistency: max diff = {diff.item()}")


def test_gradient_flow(scaler):
    """Test gradient flow"""
    device = next(scaler.parameters()).device

    x = torch.randn(2, 8, 16, 16, device=device, requires_grad=True)
    x.data = x.data * (torch.rand_like(x.data) < 0.3)

    y = scaler(x)
    loss = y.sum()
    loss.backward()

    if x.grad is None:
        raise AssertionError("No gradients computed")

    if torch.isnan(x.grad).any():
        raise AssertionError("NaN gradients detected")

    if torch.isinf(x.grad).any():
        raise AssertionError("Infinite gradients detected")


def test_memory_efficiency(scaler):
    """Test memory usage"""
    device = next(scaler.parameters()).device

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    x = torch.randn(4, 32, 64, 64, device=device)
    x = x * (torch.rand_like(x) < 0.1)

    with torch.no_grad():
        y = scaler(x)

    # Just check output is reasonable
    if torch.isnan(y).any() or torch.isinf(y).any():
        raise AssertionError("Memory efficiency test failed: NaN/Inf output")


def test_edge_cases(scaler):
    """Test edge cases"""
    device = next(scaler.parameters()).device

    edge_cases = [
        torch.zeros(2, 4, 8, 8, device=device),  # All zeros
        torch.ones(2, 4, 8, 8, device=device) * 0.5,  # All same
        torch.ones(2, 4, 8, 8, device=device) * 1e-8,  # Very small
    ]

    for i, x in enumerate(edge_cases):
        with torch.no_grad():
            y = scaler(x)

        if torch.isnan(y).any():
            raise AssertionError(f"NaN in edge case {i}")

        if torch.isinf(y).any():
            raise AssertionError(f"Inf in edge case {i}")


def test_production_stress(scaler):
    """Stress test with production-like conditions"""
    device = next(scaler.parameters()).device

    # Simulate 10 different "layers" processing
    x = torch.randn(8, 32, 64, 64, device=device)
    x = x * (torch.rand_like(x) < 0.12)  # Start very sparse

    for i in range(10):
        with torch.no_grad():
            x = scaler(x)

        # Add some noise (simulating conv operations)
        if i < 9:
            noise = torch.randn_like(x) * 0.05
            x = x + noise * (x != 0).float()

            # Gradually increase sparsity
            new_sparsity = 0.88 + i * 0.005
            keep_mask = torch.rand_like(x) < (1 - new_sparsity + 0.1)
            x = x * keep_mask.float()

        if torch.isnan(x).any() or torch.isinf(x).any():
            raise AssertionError(f"Stress test failed at iteration {i}")


def test_training_simulation(scaler):
    """Simulate training conditions"""
    device = next(scaler.parameters()).device

    # Simulate multiple training steps
    for step in range(5):
        x = torch.randn(4, 16, 32, 32, device=device, requires_grad=True)
        x.data = x.data * (torch.rand_like(x.data) < 0.15)

        y = scaler(x)
        loss = y.pow(2).sum()  # Simple loss
        loss.backward()

        if torch.isnan(x.grad).any() or torch.isinf(x.grad).any():
            raise AssertionError(f"Training simulation failed at step {step}")


# Run the comprehensive test
if __name__ == "__main__":
    best_config = comprehensive_production_test()

    print(f"\n🎯 FINAL PRODUCTION GUIDELINES:")
    print("=" * 80)
    print("1. Use the winning configuration across ALL layers")
    print("2. No per-layer parameter adjustments needed")
    print("3. Monitor for NaN/Inf during first few training epochs")
    print("4. This configuration is battle-tested for your data characteristics")
    print("5. Expected behavior: Maintains 85-89% sparsity with stable gradients")

🏭 ULTIMATE PRODUCTION VALIDATION TEST
Testing on: cpu

🧪 Testing Config 1 (Recommended)
------------------------------------------------------------
  ❌ Direction Preservation: FAILED - 
  ❌ Real-world Sparsity: FAILED - 
  ❌ Magnitude Scaling: FAILED - 
  ❌ Batch Consistency: FAILED - 
  ❌ Gradient Flow: FAILED - 
  ❌ Memory Efficiency: FAILED - 
  ❌ Edge Cases: FAILED - 
  ❌ Production Stress: FAILED - 
  ❌ Long Training Simulation: FAILED - 
  📊 Config 1 (Recommended): 0/9 tests passed (0.0%)

🧪 Testing Config 2 (Conservative)
------------------------------------------------------------
  ❌ Direction Preservation: FAILED - 
  ❌ Real-world Sparsity: FAILED - 
  ❌ Magnitude Scaling: FAILED - 
  ❌ Batch Consistency: FAILED - 
  ❌ Gradient Flow: FAILED - 
  ❌ Memory Efficiency: FAILED - 
  ❌ Edge Cases: FAILED - 
  ❌ Production Stress: FAILED - 
  ❌ Long Training Simulation: FAILED - 
  📊 Config 2 (Conservative): 0/9 tests passed (0.0%)

🧪 Testing Config 3 (Smooth)
---------------------

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VectorScaler(nn.Module):
    """Production-ready VectorScaler with fixed configuration"""

    def __init__(self,
                 act_mode: str = 'threshold',
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        norm_mags = self._normalize_preserving(magnitudes)

        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)
        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.5 + 0.7 * scale)
        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.2)
        raise ValueError(f"Invalid act_mode: {self.act_mode}")


def simple_diagnostic_test():
    """Simple test to diagnose what's happening"""
    print("🔍 SIMPLE DIAGNOSTIC TEST")
    print("=" * 50)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    # Test the recommended config
    try:
        scaler = VectorScaler(
            act_mode='threshold',
            act_threshold=0.2,
            min_magnitude=0.1
        ).to(device)
        print("✅ VectorScaler created successfully")

        # Simple test data
        x = torch.randn(2, 4, 8, 8, device=device)
        x = x * (torch.rand_like(x) < 0.15)  # Make 85% sparse
        print(f"✅ Test data created: shape={x.shape}, sparsity={((x == 0).float().mean()):.3f}")

        # Forward pass
        with torch.no_grad():
            y = scaler(x)
        print(f"✅ Forward pass successful: output shape={y.shape}")

        # Check basic properties
        input_sparsity = (x.abs() < 1e-9).float().mean()
        output_sparsity = (y.abs() < 1e-9).float().mean()
        print(f"✅ Sparsity preservation: {input_sparsity:.3f} → {output_sparsity:.3f}")

        # Check for NaN/Inf
        has_nan = torch.isnan(y).any()
        has_inf = torch.isinf(y).any()
        print(f"✅ No NaN: {not has_nan}, No Inf: {not has_inf}")

        # Test gradients
        x_grad = torch.randn(2, 4, 8, 8, device=device, requires_grad=True)
        x_grad.data = x_grad.data * (torch.rand_like(x_grad.data) < 0.15)

        y_grad = scaler(x_grad)
        loss = y_grad.sum()
        loss.backward()

        grad_ok = x_grad.grad is not None and not torch.isnan(x_grad.grad).any()
        print(f"✅ Gradient flow OK: {grad_ok}")

        # Direction preservation test
        print("\n🧭 Testing Direction Preservation:")
        x_dir = torch.zeros(2, 4, 10, 10, device=device)
        # Set up two clear directional vectors with different magnitudes
        x_dir[0, :, 5, 5] = torch.tensor([1, 0, 0, 0], device=device, dtype=x_dir.dtype)
        x_dir[0, :, 6, 6] = torch.tensor([2, 0, 0, 0], device=device, dtype=x_dir.dtype)
        x_dir[1, :, 5, 5] = torch.tensor([1, 1, 0, 0], device=device, dtype=x_dir.dtype)
        x_dir[1, :, 6, 6] = torch.tensor([3, 3, 0, 0], device=device, dtype=x_dir.dtype)

        with torch.no_grad():
            y_dir = scaler(x_dir)

        # Check a few specific vectors
        test_positions = [(0, 55), (0, 66), (1, 55), (1, 66)]  # (sample, flattened_pos)

        direction_preserved = True
        for sample, pos in test_positions:
            inp_vec = x_dir.view(2, 4, -1)[sample, :, pos]
            out_vec = y_dir.view(2, 4, -1)[sample, :, pos]

            if torch.norm(inp_vec) > 1e-6 and torch.norm(out_vec) > 1e-6:
                cos_sim = F.cosine_similarity(inp_vec.unsqueeze(0), out_vec.unsqueeze(0), dim=1)
                preserved = cos_sim.item() > 0.95
                print(f"  Sample {sample}, pos {pos}: cosine={cos_sim.item():.4f}, preserved={preserved}")
                if not preserved:
                    direction_preserved = False

        print(f"✅ Direction preservation: {direction_preserved}")

        print(f"\n🎯 DIAGNOSTIC SUMMARY:")
        print(f"✅ All basic tests passed!")
        print(f"✅ VectorScaler is working correctly with your recommended config")

        return True

    except Exception as e:
        print(f"❌ Error during diagnostic: {e}")
        import traceback
        traceback.print_exc()
        return False


def test_all_winning_configs():
    """Test all 4 winning configurations with simple checks"""
    print("\n🏆 TESTING ALL WINNING CONFIGURATIONS")
    print("=" * 60)

    configs = [
        {'act_mode': 'threshold', 'act_threshold': 0.2, 'min_magnitude': 0.1, 'name': 'Config 1'},
        {'act_mode': 'threshold', 'act_threshold': 0.3, 'min_magnitude': 0.15, 'name': 'Config 2'},
        {'act_mode': 'sigmoid', 'act_threshold': 0.2, 'min_magnitude': 0.1, 'name': 'Config 3'},
        {'act_mode': 'sigmoid', 'act_threshold': 0.25, 'min_magnitude': 0.1, 'name': 'Config 4'},
    ]

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    results = {}

    for config in configs:
        config_name = config['name']
        config_params = {k: v for k, v in config.items() if k != 'name'}

        print(f"\n🧪 Testing {config_name}: {config_params}")

        try:
            scaler = VectorScaler(**config_params).to(device)

            # Quick comprehensive test
            passed_tests = 0
            total_tests = 6

            # Test 1: Basic functionality
            x = torch.randn(2, 8, 16, 16, device=device)
            x = x * (torch.rand_like(x) < 0.12)  # 88% sparse

            with torch.no_grad():
                y = scaler(x)

            if not torch.isnan(y).any() and not torch.isinf(y).any():
                passed_tests += 1
                print("  ✅ Basic functionality")
            else:
                print("  ❌ Basic functionality - NaN/Inf detected")

            # Test 2: Sparsity preservation
            input_sparsity = (x.abs() < 1e-9).float().mean()
            output_sparsity = (y.abs() < 1e-9).float().mean()
            if abs(input_sparsity - output_sparsity) < 0.01:
                passed_tests += 1
                print("  ✅ Sparsity preservation")
            else:
                print(f"  ❌ Sparsity preservation - {input_sparsity:.3f} vs {output_sparsity:.3f}")

            # Test 3: Gradient flow
            x_grad = torch.randn(2, 8, 16, 16, device=device, requires_grad=True)
            x_grad.data = x_grad.data * (torch.rand_like(x_grad.data) < 0.15)
            y_grad = scaler(x_grad)
            loss = y_grad.sum()
            loss.backward()

            if x_grad.grad is not None and not torch.isnan(x_grad.grad).any():
                passed_tests += 1
                print("  ✅ Gradient flow")
            else:
                print("  ❌ Gradient flow")

            # Test 4: Batch consistency
            x_single = torch.randn(1, 8, 16, 16, device=device)
            x_single = x_single * (torch.rand_like(x_single) < 0.2)

            with torch.no_grad():
                y_single = scaler(x_single)
                x_batch = x_single.repeat(3, 1, 1, 1)
                y_batch = scaler(x_batch)

            max_diff = max(torch.abs(y_single[0] - y_batch[i]).max().item() for i in range(3))
            if max_diff < 1e-5:
                passed_tests += 1
                print("  ✅ Batch consistency")
            else:
                print(f"  ❌ Batch consistency - max diff: {max_diff}")

            # Test 5: Edge cases
            edge_cases_ok = True
            for case_data in [
                torch.zeros(2, 4, 8, 8, device=device),  # All zeros
                torch.ones(2, 4, 8, 8, device=device) * 0.1,  # All same small value
            ]:
                with torch.no_grad():
                    y_edge = scaler(case_data)
                if torch.isnan(y_edge).any() or torch.isinf(y_edge).any():
                    edge_cases_ok = False
                    break

            if edge_cases_ok:
                passed_tests += 1
                print("  ✅ Edge cases")
            else:
                print("  ❌ Edge cases")

            # Test 6: Multi-layer stability (stress test)
            x_stress = torch.randn(2, 16, 32, 32, device=device)
            x_stress = x_stress * (torch.rand_like(x_stress) < 0.1)  # Very sparse

            stress_ok = True
            for i in range(5):
                with torch.no_grad():
                    x_stress = scaler(x_stress)
                if torch.isnan(x_stress).any() or torch.isinf(x_stress).any():
                    stress_ok = False
                    break
                # Add some noise to simulate conv layers
                if i < 4:
                    noise = torch.randn_like(x_stress) * 0.05
                    x_stress = x_stress + noise * (x_stress != 0).float()

            if stress_ok:
                passed_tests += 1
                print("  ✅ Multi-layer stability")
            else:
                print("  ❌ Multi-layer stability")

            success_rate = passed_tests / total_tests
            results[config_name] = {
                'success_rate': success_rate,
                'passed': passed_tests,
                'total': total_tests,
                'config': config_params
            }

            print(f"  📊 {config_name}: {passed_tests}/{total_tests} passed ({success_rate:.1%})")

        except Exception as e:
            print(f"  ❌ {config_name} FAILED with error: {e}")
            results[config_name] = {
                'success_rate': 0.0,
                'passed': 0,
                'total': total_tests,
                'config': config_params,
                'error': str(e)
            }

    # Find best configuration
    print(f"\n🏆 FINAL RESULTS:")
    print("=" * 60)

    best_config = None
    best_score = -1

    for config_name, result in results.items():
        score = result['success_rate']
        print(f"{config_name}: {score:.1%} ({result['passed']}/{result['total']})")
        if score > best_score:
            best_score = score
            best_config = config_name

    if best_config and best_score > 0.8:
        winner_config = results[best_config]['config']
        print(f"\n🚀 PRODUCTION WINNER: {best_config}")
        print(f"Success rate: {best_score:.1%}")
        print(f"\n💼 PRODUCTION CODE:")
        print(f"scaler = VectorScaler(**{winner_config})")
        return winner_config
    else:
        print(f"\n⚠️  No configuration achieved >80% success rate")
        print(f"Best was {best_config} with {best_score:.1%}")
        return None


if __name__ == "__main__":
    # Run simple diagnostic first
    simple_ok = simple_diagnostic_test()

    if simple_ok:
        # Test all configurations
        winner = test_all_winning_configs()

        if winner:
            print(f"\n✅ READY FOR PRODUCTION!")
        else:
            print(f"\n⚠️  Need to investigate further")
    else:
        print(f"\n❌ Basic functionality issues - need debugging")

🔍 SIMPLE DIAGNOSTIC TEST
Device: cpu
✅ VectorScaler created successfully
✅ Test data created: shape=torch.Size([2, 4, 8, 8]), sparsity=0.873
✅ Forward pass successful: output shape=torch.Size([2, 4, 8, 8])
✅ Sparsity preservation: 0.873 → 0.873
✅ No NaN: True, No Inf: True
✅ Gradient flow OK: True

🧭 Testing Direction Preservation:
  Sample 0, pos 55: cosine=1.0000, preserved=True
  Sample 0, pos 66: cosine=1.0000, preserved=True
  Sample 1, pos 55: cosine=1.0000, preserved=True
  Sample 1, pos 66: cosine=1.0000, preserved=True
✅ Direction preservation: True

🎯 DIAGNOSTIC SUMMARY:
✅ All basic tests passed!
✅ VectorScaler is working correctly with your recommended config

🏆 TESTING ALL WINNING CONFIGURATIONS

🧪 Testing Config 1: {'act_mode': 'threshold', 'act_threshold': 0.2, 'min_magnitude': 0.1}
  ✅ Basic functionality
  ✅ Sparsity preservation
  ✅ Gradient flow
  ✅ Batch consistency
  ✅ Edge cases
  ✅ Multi-layer stability
  📊 Config 1: 6/6 passed (100.0%)

🧪 Testing Config 2: {'act_

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# CONFIGURATION VectorScaler(**{'act_mode': 'threshold', 'act_threshold': 0.2, 'min_magnitude': 0.1})
class VectorScaler(nn.Module):
    """Production-ready VectorScaler with fixed configuration"""

    def __init__(self,
                 act_mode: str = 'threshold',
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        norm_mags = self._normalize_preserving(magnitudes)

        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)
        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.5 + 0.7 * scale)
        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.2)
        raise ValueError(f"Invalid act_mode: {self.act_mode}")


def test_single_configuration_comprehensive():
    """
    PRODUCTION TEST: One configuration for ALL scenarios
    This simulates using the same VectorScaler across multiple conv layers
    """
    print("🏭 PRODUCTION READINESS TEST: Single Configuration")
    print("=" * 70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Test different possible production configurations
    candidate_configs = [
        {'act_mode': 'threshold', 'act_threshold': 0.2, 'min_magnitude': 0.1},
        {'act_mode': 'threshold', 'act_threshold': 0.3, 'min_magnitude': 0.15},
        {'act_mode': 'sigmoid', 'act_threshold': 0.2, 'min_magnitude': 0.1},
        {'act_mode': 'sigmoid', 'act_threshold': 0.25, 'min_magnitude': 0.1},
    ]

    # Comprehensive test scenarios (simulating different layers in your network)
    test_scenarios = [
        {
            'name': 'Early Layer (85% sparse)',
            'shape': (4, 16, 64, 64),
            'sparsity': 0.85,
            'magnitude_range': (0.1, 5.0)
        },
        {
            'name': 'Mid Layer (87% sparse)',
            'shape': (4, 32, 32, 32),
            'sparsity': 0.87,
            'magnitude_range': (0.05, 3.0)
        },
        {
            'name': 'Deep Layer (89% sparse)',
            'shape': (4, 64, 16, 16),
            'sparsity': 0.89,
            'magnitude_range': (0.01, 2.0)
        },
        {
            'name': '3D Volume (88% sparse)',
            'shape': (2, 8, 32, 32, 32),
            'sparsity': 0.88,
            'magnitude_range': (0.02, 4.0)
        }
    ]

    results = {}

    for config_idx, config in enumerate(candidate_configs):
        print(f"\n🧪 Testing Configuration {config_idx + 1}: {config}")
        print("-" * 50)

        scaler = VectorScaler(**config).to(device)
        config_results = {'passed': 0, 'failed': 0, 'details': []}

        for scenario in test_scenarios:
            scenario_name = scenario['name']
            print(f"  Testing: {scenario_name}")

            try:
                # Create test data matching scenario
                if len(scenario['shape']) == 5:  # 3D case
                    x = torch.randn(*scenario['shape'], device=device)
                else:  # 2D case
                    x = torch.randn(*scenario['shape'], device=device)

                # Apply sparsity
                keep_prob = 1.0 - scenario['sparsity']
                sparse_mask = torch.rand_like(x) < keep_prob
                x = x * sparse_mask.float()

                # Scale to desired magnitude range
                min_mag, max_mag = scenario['magnitude_range']
                if x[x != 0].numel() > 0:  # If there are non-zero elements
                    x_nonzero = x[x != 0]
                    current_min, current_max = x_nonzero.min(), x_nonzero.abs().max()
                    if current_max > 1e-9:
                        x = x * (max_mag / current_max)
                        x = x + (min_mag - x[x != 0].min()) * (x != 0).float()

                # Run tests
                with torch.no_grad():
                    y = scaler(x)

                # Critical checks
                checks = {
                    'no_nan': not torch.isnan(y).any(),
                    'no_inf': not torch.isinf(y).any(),
                    'sparsity_preserved': abs((x == 0).float().mean() - (y == 0).float().mean()) < 0.01,
                    'reasonable_range': y.abs().max() < 10.0,  # No exploding values
                    'direction_preserved': True  # We'll check this separately
                }

                # Direction preservation check (sample a few non-zero vectors)
                if x.numel() > 1000:  # For larger tensors, sample
                    flat_x = x.view(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
                    flat_y = y.view(y.shape[0], y.shape[1], -1).permute(0, 2, 1)

                    non_zero_mask = torch.norm(flat_x, dim=-1) > 1e-6
                    if non_zero_mask.sum() > 10:  # If we have enough non-zero vectors
                        # Sample 10 random non-zero vectors
                        non_zero_indices = torch.nonzero(non_zero_mask)[:10]
                        direction_sims = []

                        for idx in non_zero_indices:
                            i, j = idx[0], idx[1]
                            vec_in = flat_x[i, j]
                            vec_out = flat_y[i, j]

                            if torch.norm(vec_out) > 1e-6:
                                cos_sim = F.cosine_similarity(vec_in.unsqueeze(0), vec_out.unsqueeze(0), dim=1)
                                direction_sims.append(cos_sim.item())

                        if direction_sims:
                            avg_cos_sim = sum(direction_sims) / len(direction_sims)
                            checks['direction_preserved'] = avg_cos_sim > 0.95

                # Evaluate results
                all_passed = all(checks.values())
                if all_passed:
                    config_results['passed'] += 1
                    print(f"    ✅ PASSED")
                else:
                    config_results['failed'] += 1
                    failed_checks = [k for k, v in checks.items() if not v]
                    print(f"    ❌ FAILED: {failed_checks}")

                config_results['details'].append({
                    'scenario': scenario_name,
                    'passed': all_passed,
                    'checks': checks
                })

            except Exception as e:
                config_results['failed'] += 1
                print(f"    ❌ ERROR: {str(e)}")
                config_results['details'].append({
                    'scenario': scenario_name,
                    'passed': False,
                    'error': str(e)
                })

        results[config_idx] = config_results
        total_tests = config_results['passed'] + config_results['failed']
        success_rate = config_results['passed'] / total_tests if total_tests > 0 else 0
        print(f"  📊 Config {config_idx + 1} Summary: {config_results['passed']}/{total_tests} passed ({success_rate:.1%})")

    # Find best configuration
    print(f"\n🏆 PRODUCTION CONFIGURATION RECOMMENDATION")
    print("=" * 70)

    best_config_idx = -1
    best_score = -1

    for config_idx, result in results.items():
        total = result['passed'] + result['failed']
        score = result['passed'] / total if total > 0 else 0

        print(f"Configuration {config_idx + 1}: {candidate_configs[config_idx]}")
        print(f"  Success rate: {score:.1%} ({result['passed']}/{total})")

        if score > best_score:
            best_score = score
            best_config_idx = config_idx
        print()

    if best_config_idx >= 0:
        recommended_config = candidate_configs[best_config_idx]
        print(f"🚀 RECOMMENDED PRODUCTION CONFIG:")
        print(f"   VectorScaler(**{recommended_config})")
        print(f"   Success rate: {best_score:.1%}")

        return recommended_config
    else:
        print("❌ No configuration passed all tests!")
        return None


def test_recommended_config_deep_network():
    """
    Test the recommended config in a simulated deep network scenario
    """
    print(f"\n🔬 DEEP NETWORK SIMULATION TEST")
    print("=" * 50)

    # Use the most conservative config that should work well
    recommended_config = {'act_mode': 'sigmoid', 'act_threshold': 0.2, 'min_magnitude': 0.1}

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scaler = VectorScaler(**recommended_config).to(device)

    # Simulate data flowing through multiple layers
    print("Simulating data flow through 5 conv layers...")

    x = torch.randn(2, 16, 64, 64, device=device)
    # Make it 85% sparse initially
    x = x * (torch.rand_like(x) < 0.15)

    print(f"Layer 0 (input): sparsity={((x == 0).float().mean().item()):.3f}, range=[{x.min().item():.3f}, {x.max().item():.3f}]")

    for layer in range(1, 6):
        with torch.no_grad():
            # Apply VectorScaler (simulating after each conv)
            x = scaler(x)

            # Simulate some conv operation (just add some noise and change sparsity slightly)
            if layer < 5:  # Don't modify output of last layer
                conv_noise = torch.randn_like(x) * 0.1
                x = x + conv_noise * (x != 0).float()  # Only add noise to non-zero elements

                # Slightly increase sparsity as we go deeper (realistic for your data)
                new_sparsity = 0.85 + (layer * 0.01)  # 85% -> 89%
                current_nonzero = (x != 0)
                keep_mask = torch.rand_like(x) < (1 - new_sparsity + current_nonzero.float().mean())
                x = x * keep_mask.float()

        sparsity = (x == 0).float().mean().item()
        magnitude_range = [x.min().item(), x.max().item()]

        print(f"Layer {layer}: sparsity={sparsity:.3f}, range=[{magnitude_range[0]:.3f}, {magnitude_range[1]:.3f}]")

        # Check for issues
        if torch.isnan(x).any():
            print(f"❌ NaN detected at layer {layer}!")
            break
        if torch.isinf(x).any():
            print(f"❌ Inf detected at layer {layer}!")
            break
        if x.abs().max() > 20:
            print(f"⚠️  Large values detected at layer {layer}")

        if layer == 5:
            print("✅ Successfully processed through all 5 layers!")
            print(f"Final output is well-behaved and maintains sparsity structure")


if __name__ == "__main__":
    # Find best production configuration
    recommended = test_single_configuration_comprehensive()

    if recommended:
        # Test it in a deep network simulation
        test_recommended_config_deep_network()

    print(f"\n💼 PRODUCTION RECOMMENDATIONS:")
    print("=" * 50)
    print("1. Use the recommended configuration across ALL conv layers")
    print("2. Don't change thresholds between layers")
    print("3. Monitor for NaN/Inf during training")
    print("4. Consider slightly higher min_magnitude (0.15) if you see gradient issues")

🏭 PRODUCTION READINESS TEST: Single Configuration

🧪 Testing Configuration 1: {'act_mode': 'threshold', 'act_threshold': 0.2, 'min_magnitude': 0.1}
--------------------------------------------------
  Testing: Early Layer (85% sparse)
    ✅ PASSED
  Testing: Mid Layer (87% sparse)
    ✅ PASSED
  Testing: Deep Layer (89% sparse)
    ✅ PASSED
  Testing: 3D Volume (88% sparse)
    ✅ PASSED
  📊 Config 1 Summary: 4/4 passed (100.0%)

🧪 Testing Configuration 2: {'act_mode': 'threshold', 'act_threshold': 0.3, 'min_magnitude': 0.15}
--------------------------------------------------
  Testing: Early Layer (85% sparse)
    ✅ PASSED
  Testing: Mid Layer (87% sparse)
    ✅ PASSED
  Testing: Deep Layer (89% sparse)
    ✅ PASSED
  Testing: 3D Volume (88% sparse)
    ✅ PASSED
  📊 Config 2 Summary: 4/4 passed (100.0%)

🧪 Testing Configuration 3: {'act_mode': 'sigmoid', 'act_threshold': 0.2, 'min_magnitude': 0.1}
--------------------------------------------------
  Testing: Early Layer (85% sparse)
  

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import time

# Your VectorScaler implementation
class VectorScaler(nn.Module):
    """Production-ready VectorScaler with fixed configuration"""

    def __init__(self,
                 act_mode: str = 'threshold',
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        norm_mags = self._normalize_preserving(magnitudes)

        if self.act_mode != 'none':
            activated_mags = self._activate(norm_mags)
        else:
            activated_mags = norm_mags

        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        if self.act_mode == 'threshold':
            return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)
        elif self.act_mode == 'sigmoid':
            scale = torch.sigmoid((m - self.act_threshold) / (self.act_threshold + self.eps))
            return m * (0.5 + 0.7 * scale)
        elif self.act_mode == 'relu':
            return F.relu(m - self.act_threshold) + (self.act_threshold * 0.2)
        raise ValueError(f"Invalid act_mode: {self.act_mode}")


# Standard normalization approaches for comparison
class StandardBatchNorm(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.bn = nn.BatchNorm3d(num_features)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(self.bn(x))


class StandardInstanceNorm(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.instance_norm = nn.InstanceNorm3d(num_features)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(self.instance_norm(x))


class StandardLayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super().__init__()
        self.layer_norm = nn.LayerNorm(normalized_shape)
        self.activation = nn.ReLU()

    def forward(self, x):
        original_shape = x.shape
        # Apply layer norm across channel dimension
        x_flat = x.view(*x.shape[:2], -1)
        x_norm = self.layer_norm(x_flat.transpose(1, 2)).transpose(1, 2)
        x_reshaped = x_norm.view(original_shape)
        return self.activation(x_reshaped)


def create_sparse_imbalanced_data(batch_size=2, channels=16, spatial_dims=(32, 32, 32),
                                 sparsity_rate=0.85, imbalance_rate=0.98):
    """
    Creates highly sparse and imbalanced data as described in your text
    - 85% zero values
    - 97-99.5% belong to class 0 (background)
    - Remaining distributed among 3 other classes
    """
    total_voxels = np.prod(spatial_dims)

    # Create base tensor with sparse structure
    data = torch.zeros(batch_size, channels, *spatial_dims)

    for b in range(batch_size):
        # Determine non-zero locations (15% of total voxels)
        non_zero_count = int(total_voxels * (1 - sparsity_rate))

        # Randomly select non-zero locations
        flat_indices = torch.randperm(total_voxels)[:non_zero_count]

        # Convert to 3D indices
        z_indices = flat_indices // (spatial_dims[1] * spatial_dims[2])
        y_indices = (flat_indices % (spatial_dims[1] * spatial_dims[2])) // spatial_dims[2]
        x_indices = flat_indices % spatial_dims[2]

        # Fill non-zero locations with random values
        # Create class-representative patterns as mentioned in your concept
        # Class patterns: class_x (0.4, 1.3, 0.33 ratios), etc.
        for i, (z, y, x) in enumerate(zip(z_indices, y_indices, x_indices)):
            # Create different class patterns
            if i % 100 < 2:  # ~2% for class 1 (0.4x, 1.3y, 0.33z pattern)
                base_mag = torch.randn(1).abs() * 2 + 0.5
                data[b, :4, z, y, x] = base_mag * torch.tensor([0.4, 1.3, 0.33, 0.1])
                if channels > 4:
                    data[b, 4:, z, y, x] = torch.randn(channels-4) * 0.3
            elif i % 100 < 4:  # ~2% for class 2 (different pattern)
                base_mag = torch.randn(1).abs() * 2 + 0.5
                data[b, :4, z, y, x] = base_mag * torch.tensor([0.8, 0.6, 0.9, 0.2])
                if channels > 4:
                    data[b, 4:, z, y, x] = torch.randn(channels-4) * 0.3
            elif i % 100 < 5:  # ~1% for class 3
                base_mag = torch.randn(1).abs() * 2 + 0.5
                data[b, :4, z, y, x] = base_mag * torch.tensor([0.2, 0.9, 1.1, 0.4])
                if channels > 4:
                    data[b, 4:, z, y, x] = torch.randn(channels-4) * 0.3
            else:  # ~95% for class 0 (background noise)
                data[b, :, z, y, x] = torch.randn(channels) * 0.1 + 0.05

    return data


def analyze_direction_preservation(original, processed, name=""):
    """Analyze how well the processing preserves vector directions"""
    orig_flat = original.view(original.shape[0], original.shape[1], -1)
    proc_flat = processed.view(processed.shape[0], processed.shape[1], -1)

    # Calculate direction preservation for non-zero vectors
    orig_vectors = orig_flat.permute(0, 2, 1)  # (batch, spatial, channels)
    proc_vectors = proc_flat.permute(0, 2, 1)

    # Only analyze non-zero vectors
    orig_mags = torch.norm(orig_vectors, p=2, dim=-1)
    non_zero_mask = orig_mags > 1e-6

    if not non_zero_mask.any():
        return {"cosine_similarity": 0, "magnitude_ratio": 0, "analyzed_vectors": 0}

    orig_nz = orig_vectors[non_zero_mask]
    proc_nz = proc_vectors[non_zero_mask]

    # Normalize for direction comparison
    orig_nz_norm = F.normalize(orig_nz, p=2, dim=-1)
    proc_nz_norm = F.normalize(proc_nz, p=2, dim=-1)

    # Cosine similarity (direction preservation)
    cosine_sim = (orig_nz_norm * proc_nz_norm).sum(dim=-1)
    mean_cosine = cosine_sim.mean().item()

    # Magnitude changes
    orig_mag = torch.norm(orig_nz, p=2, dim=-1)
    proc_mag = torch.norm(proc_nz, p=2, dim=-1)
    mag_ratio = (proc_mag / (orig_mag + 1e-9)).mean().item()

    return {
        "cosine_similarity": mean_cosine,
        "magnitude_ratio": mag_ratio,
        "analyzed_vectors": non_zero_mask.sum().item(),
        "name": name
    }


def test_sparsity_handling(data, models):
    """Test how each model handles sparse data"""
    results = {}

    for name, model in models.items():
        with torch.no_grad():
            output = model(data)

            # Analyze sparsity preservation
            input_zeros = (data.abs() < 1e-6).sum().item()
            output_zeros = (output.abs() < 1e-6).sum().item()
            total_elements = data.numel()

            results[name] = {
                "input_sparsity": input_zeros / total_elements,
                "output_sparsity": output_zeros / total_elements,
                "sparsity_change": (output_zeros - input_zeros) / total_elements,
                "output_mean": output.mean().item(),
                "output_std": output.std().item(),
                "output_range": (output.min().item(), output.max().item())
            }

    return results


def comprehensive_test():
    """Run comprehensive comparison test"""
    print("=== VectorScaler vs Standard Normalization Comparison ===\n")

    # Create test data matching your specifications
    print("Creating sparse, imbalanced test data...")
    data = create_sparse_imbalanced_data(batch_size=4, channels=16, spatial_dims=(16, 16, 16))
    print(f"Data shape: {data.shape}")
    print(f"Sparsity: {(data.abs() < 1e-6).sum().item() / data.numel() * 100:.1f}% zeros")
    print(f"Data range: [{data.min().item():.4f}, {data.max().item():.4f}]")
    print()

    # Initialize models
    models = {
        "VectorScaler": VectorScaler(act_mode='threshold', act_threshold=0.2, min_magnitude=0.1),
        "BatchNorm+ReLU": StandardBatchNorm(16),
        "InstanceNorm+ReLU": StandardInstanceNorm(16),
        "LayerNorm+ReLU": StandardLayerNorm(16)
    }

    print("=== Processing Data ===")
    processed_outputs = {}
    processing_times = {}

    for name, model in models.items():
        start_time = time.time()
        with torch.no_grad():
            try:
                output = model(data)
                processed_outputs[name] = output
                processing_times[name] = time.time() - start_time
                print(f"✓ {name}: {processing_times[name]:.4f}s")
            except Exception as e:
                print(f"✗ {name}: Failed - {e}")
                continue

    print("\n=== Direction Preservation Analysis ===")
    for name, output in processed_outputs.items():
        analysis = analyze_direction_preservation(data, output, name)
        print(f"{name}:")
        print(f"  Cosine Similarity (direction preservation): {analysis['cosine_similarity']:.4f}")
        print(f"  Magnitude Ratio: {analysis['magnitude_ratio']:.4f}")
        print(f"  Non-zero vectors analyzed: {analysis['analyzed_vectors']}")
        print()

    print("=== Sparsity Handling Analysis ===")
    sparsity_results = test_sparsity_handling(data, models)
    for name, results in sparsity_results.items():
        print(f"{name}:")
        print(f"  Input sparsity: {results['input_sparsity']*100:.1f}%")
        print(f"  Output sparsity: {results['output_sparsity']*100:.1f}%")
        print(f"  Sparsity change: {results['sparsity_change']*100:.1f}%")
        print(f"  Output stats: mean={results['output_mean']:.4f}, std={results['output_std']:.4f}")
        print(f"  Output range: [{results['output_range'][0]:.4f}, {results['output_range'][1]:.4f}]")
        print()

    print("=== Class Pattern Analysis ===")
    # Analyze how well each method preserves class-specific patterns
    for name, output in processed_outputs.items():
        # Sample a few non-zero locations and analyze their patterns
        non_zero_mask = (data.abs() > 1e-6).any(dim=1)  # Find non-zero locations
        if non_zero_mask.any():
            # Take first batch for analysis
            sample_locations = torch.where(non_zero_mask[0])
            if len(sample_locations[0]) > 0:
                # Take first 5 locations
                n_samples = min(5, len(sample_locations[0]))
                for i in range(n_samples):
                    z, y, x = sample_locations[0][i], sample_locations[1][i], sample_locations[2][i]
                    orig_pattern = data[0, :4, z, y, x]  # First 4 channels
                    proc_pattern = output[0, :4, z, y, x]

                    # Check if this looks like a class pattern
                    if orig_pattern.abs().sum() > 0.5:  # Significant pattern
                        cosine_sim = F.cosine_similarity(orig_pattern.unsqueeze(0),
                                                       proc_pattern.unsqueeze(0))
                        if i == 0:  # Only print for first method to avoid spam
                            print(f"{name} - Sample pattern preservation:")
                        if i < 3:  # Show first 3 samples
                            print(f"  Location ({z},{y},{x}): cosine_sim = {cosine_sim.item():.4f}")

    print("\n=== Summary ===")
    print("Key findings:")
    print("1. Direction Preservation: VectorScaler should show higher cosine similarity")
    print("2. Magnitude Control: VectorScaler provides controlled magnitude scaling")
    print("3. Sparsity Handling: Compare how each method affects sparse structure")
    print("4. Pattern Preservation: Analyze preservation of class-specific patterns")

    return processed_outputs, sparsity_results


def run_gradient_flow_test():
    """Test gradient flow through different normalization methods"""
    print("\n=== Gradient Flow Test ===")

    # Create a simple test scenario
    data = create_sparse_imbalanced_data(batch_size=2, channels=8, spatial_dims=(8, 8, 8))
    data.requires_grad_(True)

    models = {
        "VectorScaler": VectorScaler(act_mode='threshold', act_threshold=0.2, min_magnitude=0.1),
        "BatchNorm+ReLU": StandardBatchNorm(8),
    }

    for name, model in models.items():
        data_copy = data.clone().detach().requires_grad_(True)

        # Forward pass
        output = model(data_copy)
        loss = output.sum()  # Simple loss for gradient computation

        # Backward pass
        loss.backward()

        # Analyze gradients
        if data_copy.grad is not None:
            grad_norm = data_copy.grad.norm().item()
            grad_mean = data_copy.grad.mean().item()
            grad_std = data_copy.grad.std().item()

            print(f"{name}:")
            print(f"  Gradient norm: {grad_norm:.6f}")
            print(f"  Gradient mean: {grad_mean:.6f}")
            print(f"  Gradient std: {grad_std:.6f}")
            print()


if __name__ == "__main__":
    # Run the comprehensive test
    outputs, sparsity_results = comprehensive_test()

    # Run gradient flow test
    run_gradient_flow_test()

    print("\n=== Test Complete ===")
    print("This test demonstrates the key differences between VectorScaler and standard approaches")
    print("on highly sparse, imbalanced data as described in your concept.")

=== VectorScaler vs Standard Normalization Comparison ===

Creating sparse, imbalanced test data...
Data shape: torch.Size([4, 16, 16, 16, 16])
Sparsity: 85.0% zeros
Data range: [-0.9347, 6.7563]

=== Processing Data ===
✓ VectorScaler: 0.0080s
✓ BatchNorm+ReLU: 0.0017s
✓ InstanceNorm+ReLU: 0.0018s
✓ LayerNorm+ReLU: 0.0032s

=== Direction Preservation Analysis ===
VectorScaler:
  Cosine Similarity (direction preservation): 1.0000
  Magnitude Ratio: 0.1405
  Non-zero vectors analyzed: 2456

BatchNorm+ReLU:
  Cosine Similarity (direction preservation): 0.8548
  Magnitude Ratio: 14.8220
  Non-zero vectors analyzed: 2456

InstanceNorm+ReLU:
  Cosine Similarity (direction preservation): 0.8551
  Magnitude Ratio: 14.8642
  Non-zero vectors analyzed: 2456

LayerNorm+ReLU:
  Cosine Similarity (direction preservation): 0.8729
  Magnitude Ratio: 6.2593
  Non-zero vectors analyzed: 2456

=== Sparsity Handling Analysis ===
VectorScaler:
  Input sparsity: 85.0%
  Output sparsity: 85.0%
  Sparsity c

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import time
# CONFIGURATION VectorScaler(**{'act_mode': 'threshold', 'act_threshold': 0.2, 'min_magnitude': 0.1})

# Your VectorScaler implementation
class VectorScaler(nn.Module):
    """Production-ready VectorScaler with fixed configuration"""

    def __init__(self,
                 act_mode: str = 'threshold',
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):
        super().__init__()
        self.act_mode = act_mode
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude

    def forward(self, x):

        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        norm_mags = self._normalize_preserving(magnitudes)


        activated_mags = self._activate(norm_mags)


        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import time

# Your VectorScaler implementation
class VectorScaler(nn.Module):
    """Production-ready VectorScaler with fixed configuration"""

    def __init__(self,
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):
        super().__init__()
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )

        norm_mags = self._normalize_preserving(magnitudes)

        activated_mags = self._activate(norm_mags)

        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import time

#  VectorScaler implementation
class VectorScaler(nn.Module):
    """Production-ready VectorScaler with fixed configuration"""

    def __init__(self,
                 act_threshold: float = 0.1,
                 eps: float = 1e-9,
                 min_magnitude: float = 0.1):
        super().__init__()
        self.act_threshold = act_threshold
        self.eps = eps
        self.min_magnitude = min_magnitude

    def forward(self, x):
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=3, dim=-1, keepdim-True)
        self_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors/self_magnitudes
        )
        original_shape = x.shape
        batch_size, channels = x.shape[:2]

        x_flat = x.view(batch_size, channels, -1)
        x_vectors = x_flat.permute(0, 2, 1)

        zero_mask = (x_vectors.abs() < self.eps).all(dim=-1, keepdim=True)

        magnitudes = torch.norm(x_vectors, p=2, dim=-1, keepdim=True)
        safe_magnitudes = magnitudes.clamp(min=self.eps)

        directions = torch.where(
            magnitudes < self.eps,
            torch.zeros_like(x_vectors),
            x_vectors / safe_magnitudes
        )
        norm_mags = self._normalize_preserving(magnitudes)

        norm_mags = self._normalize_preserving(magnitudes)

        activated_mags = self._activate(norm_mags)

        result = directions * activated_mags
        result = torch.where(zero_mask, torch.zeros_like(x_vectors), result)
        return result.permute(0, 2, 1).view(original_shape)

    def _normalize_preserving(self, magnitudes):
        non_zero_mask = magnitudes > self.eps

        non_zero_mask = magnitudes > self.eps
        if not non_zero_mask.any():
            return magnitudes

        active_mags = magnitudes[non_zero_mask]
        min_val = active_mags.min()
        max_val = active_mags.max()

        if max_val - min_val < self.eps:
            return torch.where(magnitudes > self.eps,
                             torch.full_like(magnitudes, 0.5),
                             magnitudes)

        range_val = max_val - min_val
        normalized = (magnitudes - min_val) / range_val
        scaled_range = 1.0 - self.min_magnitude
        final_normalized = self.min_magnitude + normalized * scaled_range

        return torch.where(magnitudes > self.eps, final_normalized, magnitudes)

    def _activate(self, m):
        return torch.where(m < self.act_threshold, m * 0.5, m * 1.2)



In [ ]:
x = torch.randn(1, 3, 128, 128, 128)
b, c = x.shape[:2]
print(x.shape[:2])
x_flat = x.view(b, c, -1)
x_vectors = x_flat.permute(0, 2, 1)
print(x_flat.shape)
print(x_vectors.shape)

torch.Size([1, 3])
torch.Size([1, 3, 2097152])
torch.Size([1, 2097152, 3])
